# Laboratorio Modelos de Clasificación — Ejercicio 1
## Santander Customer Satisfaction

**Plantilla de trabajo grupal para Google Colab** · Equipo: **Poma, Aldana, Brickzon, Vera y Marco**.

**Finalidad:** extender el ejercicio del docente con **4 escaladores, 5 clasificadores nuevos y 5 modelos híbridos**, implementar una comparación reproducible y escribir conclusiones. **Las celdas de código de la Parte B se dejan intencionalmente vacías.** Cada integrante debe leer la celda de instrucciones inmediatamente anterior y escribir allí su implementación.

> **Orden de lectura:** normas de trabajo → **Parte A: referencia original** → **Parte B: desarrollo grupal**. La Parte A conserva las celdas originales del notebook Santander, sin sus salidas; **no sustituye la evaluación metodológicamente corregida de la Parte B**. Para la entrega, ejecutar la Parte B de arriba hacia abajo después de completarla.

## Guía rápida de colaboración y ejecución en Colab

1. Abrir el `.ipynb` con **Google Colab**. Configurar **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Python 3**. CPU es suficiente como punto de partida; no forzar GPU. Instalar dependencias si faltan (por ejemplo: `catboost`, `xgboost`, `lightgbm`, `imbalanced-learn`).
2. Disponer de **`train.csv`**, **`test.csv`** y **`sample_submission.csv`** del ejercicio Santander. Estos CSV **no están incrustados** en este notebook. Guardarlos en Drive o subirlos a `/content`; indicar la ubicación real en la sección **4.1.1**.
3. No es necesario ejecutar las celdas de la **Parte A** para realizar la Parte B. En particular, la Parte A incluye procesamiento anterior a la división de conjuntos, una función de ROC-AUC que debe corregirse y opciones GPU que pueden no funcionar en Colab actual. Se conserva **como referencia docente**, no como fuente de métricas válidas.
4. **No usar `test.csv` de Kaggle como conjunto de evaluación**: carece de `TARGET`. El holdout se debe separar desde `train.csv` y mantenerse sin remuestreo, para medir el desempeño real.
5. Trabajar solo en las celdas asignadas. Si varias personas editan simultáneamente, acordar cómo consolidarán los cambios para evitar sobrescrituras; **Marco verifica la integración**. No cambiar los nombres de variables de la interfaz compartida sin comunicarlo.
6. Al finalizar, usar **Entorno de ejecución → Ejecutar celdas de la Parte B en orden** y comprobar que los resultados se pueden reproducir desde un entorno limpio. No entregar celdas nuevas vacías ni métricas copiadas de ejecuciones antiguas.

**Roles y alcance**

| Integrante | Bloque principal | Entregables |
|---|---|---|
| **Poma** | 4.1.2–4.1.3 y 4.2.1–4.2.4 | Datos, división sin fugas, preprocesamiento y cuatro escaladores |
| **Aldana** | 4.4.1–4.4.5 | Cinco clasificadores nuevos |
| **Brickzon** | 4.5.1–4.5.3 | Tres híbridos VotingClassifier |
| **Vera** | 4.5.4–4.5.5 y 4.6.2 | Dos híbridos StackingClassifier y verificación de ensambles |
| **Marco** | 4.1.1, 4.1.4, 4.3.1 y 4.6.1, 4.6.3–4.6.5 | Entorno Colab, función de evaluación, seis modelos base, comparación y entrega |

**Nota académica:** KNN es un **clasificador** (y también puede emplearse para imputación en otros contextos); no debe contarse como un escalador. Para cumplir el requisito de cuatro transformaciones se usarán cuatro `Scaler/Transformer` de `sklearn.preprocessing`.

# Parte A — Notebook Santander recibido del docente

Las celdas que siguen corresponden al archivo adjunto **`Santander Customer Satisfaction.ipynb`**; se mantuvo su contenido para consulta, retirando únicamente resultados/ejecuciones previas. **No modificar estas celdas como parte de las tareas nuevas.** Se incluye además el ejercicio tal como figura al final. La versión con estratificación y los notebooks de Wine Quality sirven como **referencias externas** y no se fusionan automáticamente aquí.

**Importante:** la función original `Models()` calcula ROC sobre etiquetas predichas en vez de puntuaciones, y el procesamiento original antecede a la división de entrenamiento/prueba. La implementación nueva debe resolver ambos problemas en la **Parte B**, no reutilizar los resultados originales sin corrección.

## Importar librerias

Desde equipos de soporte de primera línea hasta C-suites, la satisfacción del cliente es una medida clave del éxito. Los clientes insatisfechos no se quedan. Además, los clientes descontentos rara vez expresan su insatisfacción antes de irse.

El objetivo es identificar a los clientes insatisfechos en una etapa temprana de su relación. Hacerlo le permitiría a Santander tomar medidas proactivas para mejorar la felicidad de un cliente antes de que sea demasiado tarde.

En esta competencia, trabajará con cientos de funciones anónimas para predecir si un cliente está satisfecho o insatisfecho con su experiencia bancaria.

Se le proporciona un conjunto de datos anonimizados que contiene una gran cantidad de variables numéricas. La columna "OBJETIVO" es la variable a predecir. Es igual a uno para clientes insatisfechos y 0 para clientes satisfechos.

La tarea es predecir la probabilidad de que cada cliente en el conjunto de prueba sea un cliente insatisfecho.
Descripciones de archivos

1. train.csv: el conjunto de entrenamiento que incluye el objetivo  --- 76020
2. test.csv: el conjunto de prueba sin el objetivo  --- 75818
3. sample_submission.csv: un archivo de envío de muestra en el formato correcto

In [ ]:
#generales
import numpy as np
import pandas as pd

#gráficas
import seaborn as sns
import matplotlib.pyplot as plt

#procesamiento
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, RandomizedSearchCV

#modelo
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB

#evaluación
from scipy import stats
from sklearn.metrics import accuracy_score, roc_curve, auc, mean_squared_error, f1_score

import warnings
warnings.filterwarnings('ignore')

* Si no cuenta con alguna de las librerias ejecute el seguiente codigo según corresponda

**Instalación mencionada en la plantilla original (referencia, no ejecutar literalmente):**

```bash
pip install catboost
pip install xgboost
pip install lightgbm
pip install seaborn
pip install scipy
```

## Carga de datos

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

## 1. Análisis Exploratorio

**1.1. muestreo aleatorio del conjunto de datos**

In [ ]:
train.sample(5)

In [ ]:
test.sample(5)

In [ ]:
sample.sample(5)

**1.2. Dimención de la estructura**

In [ ]:
#determinar la dimensionalidad de los conjuntos de datos
# shape (instancias, concepto)
print("Shape de train dataframe : {}".format(train.shape))
print("Shape de test dataframe : {}".format(test.shape))
print("Shape de sample dataframe : {}".format(sample.shape))

**1.3. Descripción y estadística basica**

In [ ]:
#Comprobar la existencia de null's en los conjuntos de datos
print('Null en train : {}'.format(train.isnull().sum().any()))
print('Null en test : {}'.format(test.isnull().sum().any()))

In [ ]:
#análisis estadistico básico de las columnas numericas
train.describe()

In [ ]:
#tipos de datos
train.info()

In [ ]:
#tipos de datos
test.info()

In [ ]:
# Determinar la existencia de duplicados
dupli = train[train.duplicated()]
dupli

**1.4. Análisis Gráfico**

In [ ]:
plt.figure(figsize = (20,20))
for i in range (50):
    plt.subplot(5, 10, i+1)
    sns.boxplot(train.iloc[:,i])
    plt.xlabel(train.columns[i], size = 10)

### 2. Modificación

In [ ]:
# remover columnas con valores unicos
for i in train.columns:
    if train[i].nunique() == 1:
        print(i)
        train.drop(i, inplace = True, axis = 1)
        test.drop(i, inplace = True, axis = 1)

In [ ]:
def Outliers(data, ft):
    IQ1 = data[ft].quantile(0.25)
    IQ3 = data[ft].quantile(0.75)
    IQR = IQ3 - IQ1
    
    lower_bound = IQ1 - 1.5 * IQR
    upper_bound = IQ3 + 1.5 * IQR
    
    return lower_bound, upper_bound

In [ ]:
def replace_outliers_with_iqr(train):
    # Recorremos cada columna del DataFrame
    
    for i in train.columns:
        if i != 'TARGET':
            # Solo aplicar a columnas numéricas
            if train[i].dtype in ['int64', 'float64']:
                # Obtener los límites de los outliers
                lower_bound, upper_bound = Outliers(train, i)
                
                # Reemplazar valores que están por debajo del límite inferior
                train.loc[train[i] < lower_bound, i] = lower_bound
                
                # Reemplazar valores que están por encima del límite superior
                train.loc[train[i] > upper_bound, i] = upper_bound
            
    return train

In [ ]:
train = replace_outliers_with_iqr(train)
train

In [ ]:
print(train['TARGET'].value_counts())
plt.figure(figsize = (8,5))
sns.countplot(x = train['TARGET'])
plt.xlabel('Target', size = 12)
plt.ylabel('Count', size = 12)
plt.title('Distribución en la columna de target antes del remuestreo', size = 12)

In [ ]:
#eliminación de columnas 
test_id = test['ID']
train.drop('ID', inplace = True, axis = 1)
test.drop('ID', inplace = True, axis = 1)

**Separación de variables entrada y target**

In [ ]:
x = train.drop('TARGET', axis = 1)
x.head()

In [ ]:
y = train.loc[:, 'TARGET']
y.head()

**Técnicas de muestreo**

In [ ]:
# Muestreo de los datos, para equilibrar las clases en el target.
sampler = RandomUnderSampler(random_state=123, sampling_strategy=.5)
x, y = sampler.fit_resample(x, y)

In [ ]:
print(y.value_counts())
plt.figure(figsize = (8,5))
sns.countplot(x = y)
plt.xlabel('Target', size = 12)
plt.ylabel('Count', size = 12)
plt.title('Distribución en la columna de target despues del remuestreo', size = 12)

In [ ]:
# aplicar técnicas de escalado
# máximos minimos y standar

col = x.columns
std = StandardScaler()
x_std = std.fit_transform(x)
x_std = pd.DataFrame(data = x_std, columns = col)

mms = MinMaxScaler()
x_mms = mms.fit_transform(x)
x_mms = pd.DataFrame(data = x_mms, columns = col)

### 3. test y entrenamiento

In [ ]:
x_mms.head()

**Creación de clase model**

In [ ]:
def Models(model, name, d, xtrain, ytrain, xtest, ytest):
    print("Working on {} model".format(name))
    
    cla = model
    cla.fit(xtrain, ytrain)
    
    predicted = cla.predict(xtrain)
    tr_auc = accuracy_score(predicted, ytrain)*100
    
    predicted = cla.predict(xtest)
    te_auc = accuracy_score(predicted, ytest)*100
    
    F1 = f1_score(predicted, ytest)
    fpr, tpr, threshould = roc_curve(predicted, ytest)
    AUC = auc(fpr, tpr)
    
    d['Name'].append(name)
    d['Training ACU'].append(tr_auc)
    d['Testing ACU'].append(te_auc)
    d['F1'].append(F1)
    d['AUC'].append(AUC)
    
    print("**********"*5)
    print()
    return d, cla

**Evaluación de modelos segú el conjunto**

In [ ]:
l = [[x, 'Normal'], [x_std, 'Standard'], [x_mms, 'Min Max']]
final = []
model_fit = []
for i in l:
    xtrain, xtest, ytrain, ytest = train_test_split(i[0], y, test_size = 0.25, random_state = 42, stratify = y)
    
    d = {'Name' : [], 'Training ACU': [], 'Testing ACU': [], 'F1': [], 'AUC': []}
    
    models = [ 
        [RandomForestClassifier(n_estimators = 200), 'Random Forest'], 
        [DecisionTreeClassifier(), 'Decision Tree'], 
        [XGBClassifier(tree_method='gpu_hist'), 'XGBoost'], 
        [CatBoostClassifier(task_type="GPU"), 'CatBoost'],
        [GaussianNB(), 'Naive Bayes'], 
        [LogisticRegression(), 'Logistic Regression'] 
    ]

    for model in models:
        d, m = Models(model[0], model[1], d, xtrain, ytrain, xtest, ytest)
        model_fit.append([m, model[1], i[1]])
    final.append(d)
    

**Evaluación de los resultados por modelo y tipo de conjunto de datos**

In [ ]:
name = ['Normal', 'Standard', 'Min Max']
for i in range (len(name)):
    print(name[i])
    acu_data = pd.DataFrame(data = final[i])
    print(acu_data)
    print("******"*12)

# EJERCICIO

* Emplear cuatro modelos de escalamiento; por ejemplo:
    - robust Scaler
    - Basado en modelos (knn, etc.)

* agregar cinco modelos; por ejemplo:
    - SVM
    - AdaBoost

* preparar cinco modelos hibridos en base a los modelos previamente vistos; por ejemplo aplicando la libreria:
    - VotingClassifier

# Parte B — Desarrollo grupal: IMPLEMENTACIÓN PENDIENTE

**Cómo completar esta parte:** cada apartado incluye **(1) título, (2) celda independiente del responsable, (3) indicaciones y criterios de verificación, (4) celda de código vacía**. Si se necesita más de una celda de código, el responsable puede añadirlas debajo de la asignada. Al integrar, conservar esta estructura y escribir conclusiones en nuevas celdas Markdown.

**Contrato técnico común (no cambiar nombres sin acordarlo):**

| Nombre | Tipo y finalidad |
|---|---|
| `X_train`, `X_test` | `pandas.DataFrame` de características, con las mismas columnas y orden |
| `y_train`, `y_test` | `pandas.Series` binaria con `TARGET`: 1 = insatisfecho, 0 = satisfecho |
| `escaladores` | `dict[str, transformer]` con 4 escaladores **sin ajustar** |
| `modelos_originales` | `dict[str, classifier]` con los 6 modelos base |
| `modelos_nuevos` | `dict[str, classifier]` con los 5 modelos nuevos |
| `modelos_hibridos` | `dict[str, classifier]` con H1–H5 |
| `evaluar_modelo` | función común definida por Marco para entrenar/evaluar con el protocolo elegido |
| `resultados` | `list[dict]` o tabla consolidable, con una fila por configuración evaluada |

**Protocolo obligatorio:** `random_state=42` cuando el estimador lo permita; división **75 % entrenamiento / 25 % holdout**, estratificada; ajustar transformaciones y remuestreo **solo sobre entrenamiento**. Elegir hiperparámetros/escaladores mediante validación interna o CV sobre entrenamiento; evaluar el holdout final sin utilizarlo para decidir el modelo. Reportar mínimo Accuracy, Precision, Recall, F1 y ROC-AUC de clase 1. No remuestrear el holdout.

## 4.1. Preparación e interfaz compartida

Este bloque crea los objetos que usarán todas las secciones. Poma y Marco deben cerrar el contrato de nombres antes de que Aldana, Brickzon y Vera empiecen a integrar. Todas las celdas de implementación siguen vacías.

### 4.1.1. Preparar entorno Colab y localizar archivos

> **Responsable: Marco**

**Implementar:** instalar/importar las bibliotecas necesarias en Colab; elegir un método de acceso a datos (Google Drive o subida manual); declarar `DATA_DIR` con una ruta **real**; comprobar si existen `train.csv`, `test.csv` y `sample_submission.csv` antes de leerlos. No incrustar credenciales.

**Sugerencias de comandos (solo como guía):** `%pip install -q catboost xgboost lightgbm imbalanced-learn`. Para Drive: `from google.colab import drive` y `drive.mount('/content/drive')`. Para subida puntual: `from google.colab import files` y `files.upload()`.

**Salida esperada:** mostrar las versiones de Python y `scikit-learn`, la ruta seleccionada y la existencia de los tres CSV. Si faltan, informar claramente cuál falta.

**Verificación:** ejecutar sin rutas de Windows ni referencias a directorios locales ajenos al entorno Colab.

In [ ]:
# 4.1.1 — Entorno, dependencias, imports y rutas
from pathlib import Path
import os, sys, subprocess, importlib.util, platform

EN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or importlib.util.find_spec("google.colab") is not None
# Se instalan solo paquetes faltantes en Colab. imbalanced-learn es opcional:
# define SANTANDER_INSTALAR_IMBLEARN=1 solo si el equipo lo necesita.
_PAQUETES_COLAB = {
    "xgboost": ("xgboost", True),
    "catboost": ("catboost", True),
    "lightgbm": ("lightgbm", True),
    "imblearn": ("imbalanced-learn", os.environ.get("SANTANDER_INSTALAR_IMBLEARN", "0") == "1"),
}
if EN_COLAB:
    _faltantes = [pip_name for module_name, (pip_name, required) in _PAQUETES_COLAB.items()
                  if required and importlib.util.find_spec(module_name) is None]
    if _faltantes:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_faltantes])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report)
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

try:
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier
except ImportError as exc:
    raise ImportError("Falta XGBoost o CatBoost. En Colab, ejecuta de nuevo esta celda para instalar lo que falte.") from exc

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({"figure.figsize": (9, 5), "axes.titlesize": 13, "axes.labelsize": 11,
                     "figure.dpi": 110, "savefig.bbox": "tight"})
sns.set_theme(style="whitegrid")

DATA_DIR = Path(os.environ.get("SANTANDER_DATA_DIR", "/content"))
_nombres_csv = ("train.csv", "test.csv", "sample_submission.csv")
_candidatos = [DATA_DIR, Path("/content"), Path("/content/drive/MyDrive"), Path.cwd()]
if not all((DATA_DIR / nombre).is_file() for nombre in _nombres_csv):
    for _dir in _candidatos:
        if all((_dir / nombre).is_file() for nombre in _nombres_csv):
            DATA_DIR = _dir
            break
RUTAS_CSV = {nombre: DATA_DIR / nombre for nombre in _nombres_csv}
print(f"Google Colab: {EN_COLAB} | Python: {platform.python_version()} | scikit-learn: {sklearn.__version__}")
print(f"DATA_DIR: {DATA_DIR.resolve()}")
for _nombre, _ruta in RUTAS_CSV.items():
    print(f"{'OK' if _ruta.is_file() else 'FALTA'}: {_ruta}")
if not all(ruta.is_file() for ruta in RUTAS_CSV.values()):
    print("\nColoca los tres CSV originales en /content o en una carpeta Drive y asigna DATA_DIR a esa ruta.")
    print("Para montar Drive en Colab: from google.colab import drive; drive.mount('/content/drive')")
    print("Después define DATA_DIR = Path('/content/drive/MyDrive/Santander') y vuelve a ejecutar esta celda.")
print("Versiones: pandas", pd.__version__, "| NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)
for _mod in ("xgboost", "catboost", "lightgbm", "imblearn"):
    try:
        _spec = importlib.util.find_spec(_mod)
        print(f"{_mod}: {'disponible' if _spec else 'no instalado (opcional hasta que una sección lo requiera)'}")
    except (ImportError, ValueError):
        print(f"{_mod}: no disponible")
if EN_COLAB and importlib.util.find_spec("imblearn") is None and os.environ.get("SANTANDER_INSTALAR_IMBLEARN", "0") != "1":
    print("Si una sección necesita imbalanced-learn: define os.environ['SANTANDER_INSTALAR_IMBLEARN']='1' y vuelve a ejecutar esta celda.")


### 4.1.2. Cargar el dataset y construir la división estratificada

> **Responsable: Poma**

**Implementar:** recargar **desde los archivos originales**, sin reutilizar `train`, `x` ni `y` manipulados en la Parte A. Separar `TARGET`, retirar `ID` de los predictores, conservar `ID` aparte si luego se generan predicciones para Kaggle. Construir `X_train`, `X_test`, `y_train`, `y_test` con `train_test_split(test_size=0.25, stratify=y, random_state=42)`.

**Reglas:** `test.csv` carece de `TARGET`; **no** usarlo para métricas. No aplicar ADASYN, RandomUnderSampler, SMOTE, escaladores, clipping de valores atípicos ni selección basada en estadísticas del dataset completo **antes del split**.

**Salida esperada:** dimensiones, lista de características numéricas/no numéricas, distribución de `TARGET` antes y después del split, y confirmación de que los conjuntos no comparten registros.

**Verificación:** mismas columnas y orden para `X_train`/`X_test`; valores binarios esperados 0 y 1, proporciones estratificadas similares.

In [ ]:
# 4.1.2 Carga limpia de train.csv y holdout estratificado
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# No se reutilizan los objetos modificados en la Parte A. DATA_DIR es opcional;
# si no existe, se buscan los CSV subidos en /content o en el directorio actual.
if "DATA_DIR" in globals():
    directorios_datos = [Path(DATA_DIR)]
else:
    directorios_datos = []
directorios_datos += [Path("/content"), Path.cwd(), Path.cwd() / "Semana 13" / "santander_data"]
directorio_csv = next(
    (directorio for directorio in directorios_datos
     if (directorio / "train.csv").is_file()),
    None,
)
if directorio_csv is None:
    revisados = ", ".join(str(ruta) for ruta in directorios_datos)
    raise FileNotFoundError(
        "No se encontró train.csv. Sube train.csv a /content o define DATA_DIR. "
        f"Directorios revisados: {revisados}"
    )

ruta_train = directorio_csv / "train.csv"
train_original = pd.read_csv(ruta_train, low_memory=False)
columnas_requeridas = {"ID", "TARGET"}
if not columnas_requeridas.issubset(train_original.columns):
    raise ValueError(f"train.csv debe contener ID y TARGET; faltan: {columnas_requeridas - set(train_original.columns)}")
if train_original["TARGET"].isna().any():
    raise ValueError("TARGET contiene valores ausentes.")
clases_target = set(train_original["TARGET"].unique())
if clases_target != {0, 1}:
    raise ValueError(f"Se esperaban etiquetas binarias 0 y 1 en TARGET; se encontraron: {clases_target}")

# ID se preserva para trazabilidad, pero no se utiliza como predictor.
ID = train_original["ID"].copy()
if ID.isna().any():
    raise ValueError(f"ID contiene {int(ID.isna().sum())} valores ausentes; no se puede verificar unicidad.")
ids_duplicados = ID[ID.duplicated(keep=False)]
if not ids_duplicados.empty:
    ejemplos_ids = ids_duplicados.drop_duplicates().head(10).tolist()
    raise ValueError(
        f"ID debe ser único antes de dividir; hay {int(ID.duplicated().sum())} filas con ID repetido. "
        f"Ejemplos: {ejemplos_ids}"
    )
y = train_original["TARGET"].astype("int8").copy()
X = train_original.drop(columns=["ID", "TARGET"]).copy()
columnas_no_numericas = X.select_dtypes(exclude="number").columns.tolist()
if columnas_no_numericas:
    raise TypeError(f"Se esperaban predictores numéricos; revisar: {columnas_no_numericas}")

X_train, X_test, y_train, y_test, ID_train, ID_holdout = train_test_split(
    X, y, ID, test_size=0.25, stratify=y, random_state=42
)
if not X_train.index.intersection(X_test.index).empty:
    raise AssertionError("Hay registros compartidos entre entrenamiento y holdout.")
if list(X_train.columns) != list(X_test.columns):
    raise AssertionError("El orden de columnas no coincide entre entrenamiento y holdout.")

print(f"Fuente: {ruta_train}")
print(f"Train original: {train_original.shape}; X: {X.shape}; predictores numéricos: {X.shape[1]}")
print("Columnas identificadoras excluidas de X: ['ID']; TARGET se conserva aparte como y.")
print(f"Particiones: X_train={X_train.shape}, X_test={X_test.shape}")
print("Proporción de TARGET (train original):")
print(y.value_counts(normalize=True).sort_index().rename("proporción").to_string())
print("Conteo y proporción por partición (TARGET=0/1):")
for nombre, etiquetas in [("train", y_train), ("holdout", y_test)]:
    print(f"{nombre}: conteos={etiquetas.value_counts().sort_index().to_dict()}, "
          f"proporciones={etiquetas.value_counts(normalize=True).sort_index().round(5).to_dict()}")
print("ID_train e ID_holdout se preservan aparte; no hay filas compartidas.")
print("Valores ausentes en X_train / X_test:", int(X_train.isna().sum().sum()), "/", int(X_test.isna().sum().sum()))


### 4.1.3. Definir preprocesamiento reproducible y libre de fuga de datos

> **Responsable: Poma**

**Implementar:** proponer un `Pipeline` o una función de construcción de pipelines que ajuste **en entrenamiento** la imputación de faltantes, el escalamiento seleccionado y el clasificador. Si se eliminan columnas constantes o se limitan outliers, aprender esas reglas solo con entrenamiento y aplicar la misma transformación a holdout. Elegir y documentar una estrategia uniforme de desbalanceo (por defecto: sin remuestreo; si usan muestreo, hacerlo con un `imblearn.pipeline.Pipeline` dentro del entrenamiento y de cada fold).

**No hacer:** transformar `X_test` con un `fit_transform`, ni aplicar undersampling/SMOTE a los datos de prueba. No usar `ID` como predictor.

**Salida esperada:** objeto/fábrica reutilizable en las secciones 4.3–4.5; una breve nota de las transformaciones y su orden.

**Verificación:** `fit` ocurre solo sobre muestras de entrenamiento; transformaciones idénticas en las demás particiones.

In [ ]:
# 4.1.3 Fábrica de preprocesamiento sin fuga de datos
import warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

class EliminarColumnasVacias(BaseEstimator, TransformerMixin):
    """Detecta en fit y retira columnas completamente vacías, con aviso explícito."""
    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
            vacias = X.isna().all(axis=0).to_numpy()
        else:
            matriz = np.asarray(X, dtype=float)
            self.feature_names_in_ = np.asarray([f"x{i}" for i in range(matriz.shape[1])], dtype=object)
            vacias = np.isnan(matriz).all(axis=0)
        self.columnas_vacias_ = self.feature_names_in_[vacias].tolist()
        self.columnas_con_datos_ = self.feature_names_in_[~vacias].tolist()
        self.mascara_columnas_ = ~vacias
        if not self.mascara_columnas_.any():
            raise ValueError("Todas las columnas están vacías en el conjunto usado para fit.")
        if self.columnas_vacias_:
            warnings.warn(
                "Se excluyen columnas completamente vacías, detectadas solo en el entrenamiento: "
                f"{self.columnas_vacias_}. La misma selección se aplicará a validación/holdout.",
                UserWarning,
            )
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            faltantes = set(self.feature_names_in_) - set(X.columns)
            if faltantes:
                raise ValueError(f"Faltan columnas vistas durante fit: {sorted(faltantes)}")
            return X.loc[:, self.columnas_con_datos_]
        matriz = np.asarray(X, dtype=float)
        return matriz[:, self.mascara_columnas_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.columnas_con_datos_, dtype=object)

# El orden se conserva: detectar columnas vacías en train → imputar mediana
# aprendida en train → filtrar constantes → escalar → clasificar. No se remuestrea.
def crear_pipeline_santander(clasificador, escalador=None):
    """Crea un pipeline sin ajustar; fit debe llamarse solo con X_train/y_train."""
    pasos = [
        ("eliminar_vacias", EliminarColumnasVacias()),
        ("imputador", SimpleImputer(strategy="median")),
        ("selector_constantes", VarianceThreshold(threshold=0.0)),
    ]
    if escalador is not None:
        pasos.append(("escalador", clone(escalador)))
    pasos.append(("clasificador", clone(clasificador)))
    return Pipeline(steps=pasos)

# Para mostrar escaladores sin entrenar modelos, se usa la misma secuencia
# de preprocesamiento de arriba, ajustada exclusivamente sobre X_train.
def demostrar_escalador_en_train(escalador, columnas=None):
    if columnas is None:
        candidatas = X_train.select_dtypes(include="number").columns.tolist()
    else:
        candidatas = list(columnas)
    # Se eligen columnas con al menos dos valores observados distintos en train;
    # así la imputación de mediana no puede dejar una entrada constante al selector.
    columnas = [
        columna for columna in candidatas
        if X_train[columna].nunique(dropna=True) > 1
    ][:8]
    if not columnas:
        raise ValueError("No hay columnas numéricas con variación en X_train para la demostración.")
    preprocesamiento_demo = Pipeline([
        ("eliminar_vacias", EliminarColumnasVacias()),
        ("imputador", SimpleImputer(strategy="median")),
        ("selector_constantes", VarianceThreshold(threshold=0.0)),
    ])
    train_preparado = preprocesamiento_demo.fit_transform(X_train[columnas])
    columnas_preparadas = preprocesamiento_demo.named_steps["selector_constantes"].get_feature_names_out(
        preprocesamiento_demo.named_steps["imputador"].get_feature_names_out(
            preprocesamiento_demo.named_steps["eliminar_vacias"].get_feature_names_out()
        )
    )
    transformador_demo = clone(escalador)
    train_escalado = transformador_demo.fit_transform(train_preparado)
    return train_escalado, list(columnas_preparadas)

print("Pipeline definido: detectar vacías → imputar mediana → filtrar constantes → escalador opcional → clasificador.")
print("El detector informa explícitamente columnas totalmente vacías y ajusta su selección solo con train.")
print("Sin remuestreo: toda transformación aprende al ejecutar fit con X_train/y_train.")
print("Para holdout, usar pipeline.predict(X_test); nunca ajustar ni transformar con fit sobre X_test.")


### 4.1.4. Definir la interfaz de evaluación y resultados

> **Responsable: Marco**

**Implementar:** una función `evaluar_modelo(...)` (o interfaz equivalente, documentando argumentos) que reciba nombre, tipo (`original`, `nuevo` o `hibrido`), clasificador y escalador; construya el pipeline adecuado, entrene en `X_train/y_train` y calcule métricas sobre validación/holdout según el protocolo. Registrar tiempo de entrenamiento y resultados en una **estructura común**, sin cambiar su forma entre integrantes.

**Columnas obligatorias de la tabla final:** `tipo`, `modelo`, `escalador`, `accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `tiempo_s`. Agregar `etapa` (`validacion`/`holdout`) y `semilla` si conviene.

**ROC-AUC correcto:** usar `roc_auc_score(y_real, proba_clase_1)` o `decision_function`; `roc_curve(y_real, puntaje_clase_1)` para curvas ROC. **No** usar `roc_curve(y_pred, y_real)` como en la función original. Controlar divisiones por cero al calcular Precision/Recall/F1. Nunca decidir el mejor modelo basándose reiteradamente en el holdout.

**Validación:** prueba de la interfaz con un clasificador sencillo sobre una partición de validación interna sin modificar el test final; conservar las salidas compatibles con los tres diccionarios de modelos.

In [ ]:
# 4.1.4 — Evaluación común. Las particiones son obligatorias y su procedencia se verifica.
COLUMNAS_RESULTADOS = ["tipo", "modelo", "escalador", "accuracy", "precision", "recall", "f1",
    "roc_auc", "tiempo_s", "etapa", "semilla", "roc_auc_motivo", "matriz_confusion", "fold"]
resultados = []
tabla_resultados = pd.DataFrame(columns=COLUMNAS_RESULTADOS)
tabla_validacion_ultima = pd.DataFrame()
modelos_holdout, modelos_validacion, predicciones_holdout = {}, {}, {}


# Contrato compartido de Poma (4.1.3):
# crear_pipeline_santander(clasificador, escalador=None) devuelve un Pipeline
# sklearn completo, sin ajustar, que incluye transformaciones y clasificador.
# Marco lo consume directamente: no añade un segundo preprocesador ni sampler.
CONTRATO_PREPROCESAMIENTO_COMPARTIDO = {
    "factory": "crear_pipeline_santander(clasificador, escalador=None)",
    "retorno": "Pipeline sklearn completo, sin ajustar, con clasificador final.",
    "remuestreo": "No incluido; se conserva la decisión actual de Poma de no remuestrear.",
}
_PREPROCESAMIENTO_ULTIMO_ORIGEN = None
_HOLDOUT_GATE = object()
_HOLDOUT_ACTIVE_TOKEN = None

def _crear_preprocesador_local(X, escalador=None, denso=False):
    numericas = X.select_dtypes(include=[np.number]).columns.tolist()
    categoricas = [c for c in X.columns if c not in numericas]
    pasos = []
    if numericas:
        numeric = Pipeline([("imputador", SimpleImputer(strategy="median"))])
        if escalador is not None:
            numeric.steps.append(("escalador", clone(escalador)))
        pasos.append(("numericas", numeric, numericas))
    if categoricas:
        categorical = Pipeline([("imputador", SimpleImputer(strategy="most_frequent")),
                                ("onehot", OneHotEncoder(handle_unknown="ignore"))])
        pasos.append(("categoricas", categorical, categoricas))
    if not pasos:
        raise ValueError("No hay predictores utilizables.")
    return ColumnTransformer(pasos, remainder="drop", sparse_threshold=0 if denso else 0.3)

def _crear_pipeline_evaluacion(clasificador, X, escalador=None):
    """Usa el pipeline completo de Poma; fallback local solo si la fábrica falta."""
    global _PREPROCESAMIENTO_ULTIMO_ORIGEN
    if not isinstance(X, pd.DataFrame):
        raise TypeError("X debe ser pandas.DataFrame para conservar índices y columnas.")
    factory = globals().get("crear_pipeline_santander")
    if factory is not None:
        if not callable(factory):
            raise TypeError("crear_pipeline_santander debe ser callable según el contrato de Poma.")
        try:
            modelo = factory(clone(clasificador),
                escalador=clone(escalador) if escalador is not None else None)
        except TypeError as exc:
            raise TypeError("La fábrica de Poma debe aceptar (clasificador, escalador=None).") from exc
        if not isinstance(modelo, Pipeline):
            raise TypeError("crear_pipeline_santander debe devolver un sklearn Pipeline completo.")
        if not modelo.steps or modelo.steps[-1][0] != "clasificador":
            raise ValueError("El pipeline compartido debe terminar con el paso 'clasificador'.")
        nombres_escaladores = [nombre for nombre, _ in modelo.steps if nombre == "escalador"]
        if len(nombres_escaladores) != (1 if escalador is not None else 0):
            raise ValueError("El pipeline compartido debe incluir el escalador solicitado exactamente una vez.")
        try:
            modelo = clone(modelo)
        except Exception as exc:
            raise TypeError("El pipeline compartido de Poma debe ser clonable y no estar ajustado.") from exc
        _PREPROCESAMIENTO_ULTIMO_ORIGEN = "Poma: crear_pipeline_santander (pipeline completo)"
        return modelo

    warnings.warn(
        "crear_pipeline_santander no está disponible; se utiliza el fallback local de Marco. "
        "Este fallback mantiene imputación mediana, escalado opcional y codificación categórica; "
        "no incorpora remuestreo.", UserWarning)
    preprocesador = _crear_preprocesador_local(X, escalador, isinstance(clasificador, GaussianNB))
    _PREPROCESAMIENTO_ULTIMO_ORIGEN = "fallback local de Marco (Poma no disponible)"
    return Pipeline([("preprocesamiento", preprocesador), ("clasificador", clone(clasificador))])

def _autorizar_holdout_controlado():
    state = globals().get("ESTADO_COMPARACION", {})
    active = globals().get("_HOLDOUT_ACTIVE_TOKEN")
    if active is not _HOLDOUT_GATE:
        raise RuntimeError("El holdout solo se evalúa mediante evaluar_holdout_final().")
    if state.get("holdout_status") != "evaluando" or state.get("evaluada_revision") != state.get("revision"):
        raise RuntimeError("No hay una selección CV vigente autorizada para evaluar el holdout.")


def _filas_coinciden(particion, referencia, nombre):
    try:
        esperado = referencia.loc[particion.index]
    except KeyError as exc:
        raise ValueError(f"{nombre} contiene índices ajenos a su partición de origen.") from exc
    if not particion.equals(esperado):
        raise ValueError(f"Los datos de {nombre} no corresponden a las filas originales de su partición.")

def _validar_particiones(X_fit, y_fit, X_eval, y_eval, etapa):
    if etapa not in {"validacion", "holdout"}:
        raise ValueError("etapa debe ser 'validacion' o 'holdout'.")
    for name, value, expected in (("X_fit", X_fit, pd.DataFrame), ("X_eval", X_eval, pd.DataFrame),
                                  ("y_fit", y_fit, pd.Series), ("y_eval", y_eval, pd.Series)):
        if not isinstance(value, expected):
            raise TypeError(f"{name} debe ser {expected.__name__}; pasa las particiones explícitamente.")
    if not X_fit.index.is_unique or not X_eval.index.is_unique:
        raise ValueError("Los índices de las particiones deben ser únicos.")
    if not y_fit.index.equals(X_fit.index) or not y_eval.index.equals(X_eval.index):
        raise ValueError("Cada y debe conservar exactamente los índices y orden de su X.")
    if list(X_fit.columns) != list(X_eval.columns):
        raise ValueError("X_fit y X_eval deben tener las mismas columnas y orden.")
    if "ID" in X_fit.columns:
        raise ValueError("ID no puede ser predictor.")
    Xtr, Xte = globals().get("X_train"), globals().get("X_test")
    ytr, yte = globals().get("y_train"), globals().get("y_test")
    if not isinstance(Xtr, pd.DataFrame) or not isinstance(ytr, pd.Series):
        raise ValueError("Ejecuta el split de Poma para comprobar la procedencia de las particiones.")
    if not Xtr.index.is_unique or not ytr.index.equals(Xtr.index):
        raise ValueError("X_train/y_train tienen índices repetidos o desalineados.")
    if etapa == "validacion":
        if X_fit.index.intersection(X_eval.index).size:
            raise ValueError("Entrenamiento y validación se solapan.")
        if not X_fit.index.isin(Xtr.index).all() or not X_eval.index.isin(Xtr.index).all():
            raise ValueError("Validación solo acepta folds derivados de X_train; X_test/y_test están prohibidos.")
        _filas_coinciden(X_fit, Xtr, "X_fit")
        _filas_coinciden(y_fit, ytr, "y_fit")
        _filas_coinciden(X_eval, Xtr, "X_eval")
        _filas_coinciden(y_eval, ytr, "y_eval")
        if isinstance(Xte, pd.DataFrame) and Xtr.index.intersection(Xte.index).size:
            raise ValueError("El split compartido presenta índices entre train y holdout.")
    else:
        if not isinstance(Xte, pd.DataFrame) or not isinstance(yte, pd.Series):
            raise ValueError("Holdout requiere X_test/y_test definidos por Poma.")
        if not Xte.index.is_unique or not yte.index.equals(Xte.index):
            raise ValueError("X_test/y_test tienen índices repetidos o desalineados.")
        if Xtr.index.intersection(Xte.index).size:
            raise ValueError("El split compartido presenta índices entre train y holdout.")
        if not X_fit.equals(Xtr) or not y_fit.equals(ytr):
            raise ValueError("Holdout se ajusta con el X_train/y_train completos.")
        if not X_eval.equals(Xte) or not y_eval.equals(yte):
            raise ValueError("Holdout evalúa exactamente los datos/etiquetas originales reservados.")

def _publicar_resultados():
    tabla = pd.DataFrame(resultados)
    globals()["tabla_resultados"] = (tabla.reindex(columns=COLUMNAS_RESULTADOS) if not tabla.empty
                                      else pd.DataFrame(columns=COLUMNAS_RESULTADOS))
    return globals()["tabla_resultados"]

def _puntaje_continuo(ajustado, X_eval):
    if hasattr(ajustado, "predict_proba"):
        proba = np.asarray(ajustado.predict_proba(X_eval))
        clases = list(getattr(ajustado, "classes_", []))
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, clases.index(1) if 1 in clases else 1]
    if hasattr(ajustado, "decision_function"):
        score = np.asarray(ajustado.decision_function(X_eval))
        clases = list(getattr(ajustado, "classes_", []))
        if score.ndim == 1:
            return score if len(clases) < 2 or clases[-1] == 1 else -score
        return score[:, clases.index(1) if 1 in clases else 1]
    return None

def _resolver_nombre_escalador(escalador):
    if escalador is None:
        return "sin_escalado"
    for clave, objeto in globals().get("escaladores", {}).items():
        if objeto is escalador:
            return clave
    return type(escalador).__name__

def evaluar_modelo(nombre, tipo, clasificador, escalador=None, *, X_fit, y_fit, X_eval, y_eval,
                   etapa="validacion", guardar=True, guardar_ajustado=False,
                   calcular_matriz=False, semilla=RANDOM_STATE, fold=None, nombre_escalador=None):
    """Entrena solo en X_fit; exige argumentos explícitos y comprueba la procedencia."""
    if tipo not in {"original", "nuevo", "hibrido"}:
        raise ValueError("tipo debe ser 'original', 'nuevo' o 'hibrido'.")
    if etapa == "holdout":
        _autorizar_holdout_controlado()
    _validar_particiones(X_fit, y_fit, X_eval, y_eval, etapa)
    modelo = _crear_pipeline_evaluacion(clasificador, X_fit, escalador)
    import time
    inicio = time.perf_counter()
    modelo.fit(X_fit, y_fit)
    tiempo = time.perf_counter() - inicio
    pred, y_real = np.asarray(modelo.predict(X_eval)), np.asarray(y_eval)
    score = _puntaje_continuo(modelo, X_eval)
    auc = np.nan
    if score is not None and len(np.unique(y_real)) == 2:
        try:
            auc = float(roc_auc_score(y_real, score))
        except ValueError:
            pass
    registro = {"tipo": tipo, "modelo": nombre, "escalador": nombre_escalador or _resolver_nombre_escalador(escalador),
        "accuracy": float(accuracy_score(y_real, pred)),
        "precision": float(precision_score(y_real, pred, pos_label=1, zero_division=0)),
        "recall": float(recall_score(y_real, pred, pos_label=1, zero_division=0)),
        "f1": float(f1_score(y_real, pred, pos_label=1, zero_division=0)), "roc_auc": auc,
        "tiempo_s": float(tiempo), "etapa": etapa, "semilla": semilla,
        "roc_auc_motivo": None if score is not None else "Sin predict_proba ni decision_function.",
        "matriz_confusion": confusion_matrix(y_real, pred, labels=[0, 1]).tolist() if calcular_matriz else None,
        "fold": fold}
    clave = (tipo, nombre, registro["escalador"], etapa, fold)
    if guardar:
        resultados.append(registro)
        _publicar_resultados()
    if guardar_ajustado and etapa == "validacion":
        modelos_validacion[clave] = modelo
    if etapa == "holdout":
        clave_final = (tipo, nombre, registro["escalador"])
        predicciones_holdout[clave_final] = {"y_true": y_real, "y_pred": pred, "score": score}
        if guardar_ajustado:
            modelos_holdout[clave_final] = modelo
    return registro

## 4.2. Cuatro métodos de escalamiento

Crear el diccionario `escaladores` con **exactamente** estas cuatro entradas: `standard`, `minmax`, `robust`, `quantile`. **No ajustar** los transformadores al crearlos: dejarlos listos para integrarse en los pipelines y ajustar con `fit` exclusivamente sobre el entrenamiento. Comparar qué cambia en distribución/rangos sin usar información del holdout.

### 4.2.1. StandardScaler

> **Responsable: Poma**

**Implementar:** incorporar `StandardScaler` al diccionario `escaladores` con la clave `standard`. Explicar en Markdown la estandarización mediante media y desviación típica.

**Verificación:** comprobar en un subconjunto **de entrenamiento transformado** media aproximadamente 0 y desviación aproximadamente 1 para columnas no constantes; no ajustar al holdout.

**Entregable:** transformador registrable y una interpretación breve.

In [ ]:
# 4.2.1 StandardScaler: centra con la media y escala por desviación estándar
from sklearn.preprocessing import StandardScaler

# Las cuatro demostraciones usan demostrar_escalador_en_train(), que replica
# la imputación, el filtro de constantes y el ajuste únicamente en X_train.
# Las instancias registradas permanecen sin ajustar para la fábrica de pipelines.
escaladores = {"standard": StandardScaler()}
train_standard_demo, columnas_standard_demo = demostrar_escalador_en_train(escaladores["standard"])
print("Escalador registrado (sin ajustar): standard")
print("Demostración independiente, con preprocesamiento compatible sobre X_train.")
print("Columnas variables conservadas:", columnas_standard_demo)
print("Media y desviación poblacional tras estandarizar:")
print("media:", train_standard_demo.mean(axis=0).round(3))
print("desv. estándar:", train_standard_demo.std(axis=0).round(3))


### 4.2.2. MinMaxScaler

> **Responsable: Poma**

**Implementar:** incorporar `MinMaxScaler` con clave `minmax`, rango de entrenamiento `[0, 1]` (a menos que documenten otra elección).

**Verificación:** revisar los mínimos y máximos del **entrenamiento transformado**; valores del holdout pueden quedar fuera de `[0,1]` si exceden el rango aprendido, y eso no debe corregirse con un reajuste.

**Entregable:** transformador y breve interpretación.

In [ ]:
# 4.2.2 MinMaxScaler: transforma usando los mínimos y máximos de entrenamiento
from sklearn.preprocessing import MinMaxScaler

escaladores["minmax"] = MinMaxScaler(feature_range=(0, 1))
train_minmax_demo, columnas_minmax_demo = demostrar_escalador_en_train(escaladores["minmax"])
print("Escalador registrado (sin ajustar): minmax")
print("Demostración independiente, con preprocesamiento compatible sobre X_train.")
print("Mínimos de X_train transformado:", train_minmax_demo.min(axis=0).round(3))
print("Máximos de X_train transformado:", train_minmax_demo.max(axis=0).round(3))
print("Columnas variables conservadas:", columnas_minmax_demo)
print("El holdout puede quedar fuera de [0, 1] si supera los extremos vistos en train; no se reajusta.")


### 4.2.3. RobustScaler

> **Responsable: Poma**

**Implementar:** incorporar `RobustScaler` con clave `robust`; explicar por qué mediana y rango intercuartílico reducen la influencia de valores extremos.

**Verificación:** comparar con StandardScaler usando ejemplos de características con outliers del **entrenamiento**. No eliminar automáticamente valores atípicos sin justificarlo.

**Entregable:** transformador y breve interpretación.

In [ ]:
# 4.2.3 RobustScaler: usa mediana y rango intercuartílico
from sklearn.preprocessing import RobustScaler, StandardScaler

escaladores["robust"] = RobustScaler()
train_robust_demo, columnas_robust_demo = demostrar_escalador_en_train(escaladores["robust"])
train_standard_comparacion, _ = demostrar_escalador_en_train(StandardScaler())
print("Escalador registrado (sin ajustar): robust")
print("Demostración independiente, con preprocesamiento compatible sobre X_train.")
print("Medianas tras RobustScaler:", np.median(train_robust_demo, axis=0).round(3))
print("La mediana y el rango intercuartílico son menos sensibles a valores extremos que media/desviación estándar.")
print("Desviaciones estándar de la misma muestra procesada:")
print("Standard:", train_standard_comparacion.std(axis=0).round(3))
print("Robust:", train_robust_demo.std(axis=0).round(3))
print("Columnas variables conservadas:", columnas_robust_demo)
print("No se eliminan ni recortan outliers en este preprocesamiento.")


### 4.2.4. QuantileTransformer

> **Responsable: Poma**

**Implementar:** incorporar `QuantileTransformer` con clave `quantile`, `output_distribution='normal'`, `random_state=42` y un `n_quantiles` no mayor que el tamaño de entrenamiento usado para ajustar.

**Verificación:** confirmar transformación de distribuciones en una muestra de entrenamiento; explicar que este método puede cambiar distancias/relaciones y por eso debe compararse empíricamente.

**Entregable:** cuarto transformador; mostrar `list(escaladores)` con **exactamente cuatro** claves distintas.

In [ ]:
# 4.2.4 QuantileTransformer: lleva cada marginal a una distribución normal
from sklearn.preprocessing import QuantileTransformer

n_cuantiles = min(1000, len(X_train))
escaladores["quantile"] = QuantileTransformer(
    n_quantiles=n_cuantiles,
    output_distribution="normal",
    random_state=42,
)
train_quantile_demo, columnas_quantile_demo = demostrar_escalador_en_train(escaladores["quantile"])
print("Escalador registrado (sin ajustar): quantile")
print("Demostración independiente, con preprocesamiento compatible sobre X_train.")
print(f"n_quantiles={n_cuantiles}; tamaño de X_train={len(X_train)}")
print("Media de la muestra transformada:", train_quantile_demo.mean(axis=0).round(3))
print("Columnas variables conservadas:", columnas_quantile_demo)
print("El método transforma las distribuciones marginales y puede cambiar distancias/relaciones; su efecto se compara empíricamente.")
print("Escaladores disponibles:", list(escaladores))
if set(escaladores) != {"standard", "minmax", "robust", "quantile"}:
    raise AssertionError("El registro debe contener exactamente los cuatro escaladores acordados.")


## 4.3. Seis clasificadores originales como línea base

### 4.3.1. Reconstruir los seis modelos de la clase bajo el nuevo protocolo

> **Responsable: Marco**

**Implementar:** registrar en `modelos_originales` estos seis clasificadores utilizados en el notebook Santander del docente: `random_forest`, `decision_tree`, `xgboost`, `catboost`, `gaussian_nb`, `logistic_regression`. Usar semillas cuando corresponda y una configuración razonable para Colab.

**Precaución:** el notebook original fuerza GPU en XGBoost/CatBoost y podría fallar en el entorno actual. Para esta plantilla, preferir configuraciones CPU compatibles; solo activar GPU cuando se haya verificado disponibilidad. Evitar copiar la función original `Models()` por su implementación incorrecta de ROC.

**Comparación:** entrenar y evaluar las líneas base con la interfaz común, sin mezclar métricas del notebook original con las de esta Parte B. Para cuantificar el efecto del escalamiento, usar los cuatro transformadores en condiciones comparables; si el costo impide evaluar algunas combinaciones, justificar explícitamente qué se ejecutó y qué no.

**Salida esperada:** `len(modelos_originales) == 6` y resultados trazables con el mismo split/semilla/métricas.

In [ ]:
# 4.3.1 — Seis clasificadores originales; solo se construyen, no se entrenan aquí.
modelos_originales = {
    "random_forest": RandomForestClassifier(n_estimators=250, min_samples_leaf=2,
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE),
    "decision_tree": DecisionTreeClassifier(max_depth=12, min_samples_leaf=10,
        class_weight="balanced", random_state=RANDOM_STATE),
    "xgboost": XGBClassifier(n_estimators=250, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE),
    "catboost": CatBoostClassifier(iterations=250, depth=6, learning_rate=0.05,
        loss_function="Logloss", verbose=False, allow_writing_files=False,
        thread_count=-1, random_seed=RANDOM_STATE),
    "gaussian_nb": GaussianNB(),
    "logistic_regression": LogisticRegression(C=1.0, max_iter=1500,
        class_weight="balanced", solver="liblinear", random_state=RANDOM_STATE)
}
assert len(modelos_originales) == 6 and len(set(modelos_originales)) == 6
print("Modelos originales listos (sin entrenar):", list(modelos_originales))

## 4.4. Cinco clasificadores adicionales

Los **cinco modelos de este apartado son nuevos respecto a la lista de seis modelos entrenados en la Parte A**. Registrarlos en `modelos_nuevos`, usando el mismo protocolo de evaluación de 4.1.4. No reescribir el split ni ajustar escaladores sobre todo el dataset. **Aldana** crea el diccionario en 4.4.1 y lo completa en los siguientes apartados.

### 4.4.1. SVM de clasificación (LinearSVC)

> **Responsable: Aldana**

**Implementar:** `LinearSVC` como variante lineal de SVM para controlar el costo computacional del dataset Santander; registrar con clave `svm_lineal`. Usar escalamiento e incrementar iteraciones solo si es necesario, explicando convergencia y margen.

**ROC-AUC:** `LinearSVC` no expone `predict_proba` de forma nativa; usar `decision_function` para ROC-AUC. No incluirlo directamente en híbridos de `voting='soft'` sin calibrar probabilidades.

**Entregable:** entrenamiento, métricas comparables e interpretación del comportamiento de SVM.

In [ ]:
# 4.4.1 — SVM lineal, sin ajustar todavía.
from sklearn.svm import LinearSVC

# Este bloque inicia el registro acordado; ejecutar 4.4.1–4.4.5 en orden.
# dual=False es apropiado cuando Santander tiene más filas que predictores.
# El margen se aprende en el pipeline con el escalador de cada comparación.
# No se suprimen ConvergenceWarning: si aparecen, revisar la convergencia
# usando únicamente entrenamiento antes de aumentar max_iter o cambiar C.
modelos_nuevos = {
    "svm_lineal": LinearSVC(
        C=1.0, dual=False, tol=1e-4, max_iter=10000, random_state=42,
    )
}
# LinearSVC aporta decision_function para ROC-AUC; no probabilidades.
# Se puede usar en Stacking, pero Voting Soft requeriría calibración dentro de CV.
print("Registrado svm_lineal: margen lineal; ROC-AUC mediante decision_function.")


### 4.4.2. AdaBoostClassifier

> **Responsable: Aldana**

**Implementar:** `AdaBoostClassifier` como `adaboost`, con `random_state=42`; elegir un número razonable de estimadores y documentarlo.

**Verificación:** registrar Accuracy, Precision, Recall, F1 y ROC-AUC; comparar resultados con Decision Tree, su posible clasificador base.

**Entregable:** modelo registrable, métricas e interpretación.

In [ ]:
# 4.4.2 — AdaBoost con 100 árboles de profundidad 1.
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

modelos_nuevos["adaboost"] = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100, learning_rate=0.5, random_state=42,
)
# Los stumps se entrenan secuencialmente, dando más peso a errores previos.
# 100 iteraciones y learning_rate=0.5 son valores iniciales, no un óptimo medido.
# La evaluación común comparará AdaBoost con decision_tree en los mismos folds.
print("Registrado adaboost: 100 stumps, learning_rate=0.5, semilla=42.")


### 4.4.3. KNeighborsClassifier (KNN)

> **Responsable: Aldana**

**Implementar:** `KNeighborsClassifier` como `knn`. Elegir un valor inicial de `n_neighbors` y, si hay tiempo, compararlo mediante validación **solo sobre entrenamiento**.

**Precaución:** KNN es sensible a la escala, dimensión y costo de distancia; no contarlo como uno de los cuatro escaladores y evitar `KNNImputer` masivo salvo necesidad y justificación.

**Entregable:** modelo registrable y explicación de qué transformador le favoreció.

In [ ]:
# 4.4.3 — KNN con 11 vecinos y distancia euclídea.
from sklearn.neighbors import KNeighborsClassifier

modelos_nuevos["knn"] = KNeighborsClassifier(
    n_neighbors=11, weights="distance", metric="minkowski", p=2,
    algorithm="brute", n_jobs=-1,
)
# k=11 es un punto de partida fijo, no elegido con el holdout.
# KNN necesita escalamiento: se compara con los cuatro transformadores
# ajustados dentro de cada fold. En alta dimensión las distancias pueden
# discriminar menos y la predicción puede ser costosa; brute evita asumir
# que un árbol de búsqueda acelerará las distancias en cientos de columnas.
print("Registrado knn: k=11, pesos por distancia; comparar los cuatro escaladores.")


### 4.4.4. LightGBMClassifier

> **Responsable: Aldana**

**Implementar:** `LGBMClassifier` como `lightgbm`, con semilla fija y parámetros moderados para Colab. Aunque la biblioteca aparezca en las **importaciones** originales, **no figura entre los seis clasificadores entrenados**, así que cuenta como modelo adicional.

**Verificación:** métricas sobre particiones acordadas y comparación con XGBoost/CatBoost.

**Entregable:** modelo registrado, configuración y evaluación.

In [ ]:
# 4.4.4 — LightGBM con configuración moderada para CPU.
try:
    from lightgbm import LGBMClassifier
except ImportError as exc:
    raise ImportError(
        "LightGBM es obligatorio para 4.4.4. Completa la instalación en 4.1.1 "
        "y vuelve a ejecutar este bloque; no se sustituye por otro clasificador."
    ) from exc

modelos_nuevos["lightgbm"] = LGBMClassifier(
    objective="binary", n_estimators=200, learning_rate=0.05,
    num_leaves=31, max_depth=-1, min_child_samples=20,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    device_type="cpu", n_jobs=2, random_state=42, verbosity=-1,
)
# Boosting por histogramas; subsample_freq activa el muestreo de filas.
# No usar el holdout para early stopping ni para ajustar hiperparámetros.
# La comparación con XGBoost/CatBoost utiliza la misma interfaz y los mismos folds.
print("Registrado lightgbm: 200 árboles, 31 hojas, CPU y semilla=42.")


### 4.4.5. ExtraTreesClassifier

> **Responsable: Aldana**

**Implementar:** `ExtraTreesClassifier` como `extra_trees`; fijar `random_state=42` y documentar número de árboles.

**Verificación:** comparar su comportamiento con Random Forest y explicar el mayor grado de aleatoriedad de Extra Trees.

**Entregable:** modelo registrado, métricas y comprobación final de `len(modelos_nuevos) == 5`.

In [ ]:
# 4.4.5 — Extra Trees con 250 árboles.
from sklearn.ensemble import ExtraTreesClassifier

modelos_nuevos["extra_trees"] = ExtraTreesClassifier(
    n_estimators=250, min_samples_leaf=2, max_features="sqrt",
    bootstrap=False, n_jobs=-1, random_state=42,
)
# Extra Trees aleatoriza umbrales de división además de las características.
# Random Forest busca el mejor umbral entre las características muestreadas;
# la mayor aleatoriedad no garantiza mayor calidad: se contrasta mediante CV.
# Se conserva el desbalance original en los cinco modelos: sin remuestreo
# ni pesos de clase adicionales. La Accuracy se interpreta junto con Recall,
# Precision, F1 y ROC-AUC para TARGET=1 (cliente insatisfecho).
_claves_nuevas = {"svm_lineal", "adaboost", "knn", "lightgbm", "extra_trees"}
assert len(modelos_nuevos) == 5 and set(modelos_nuevos) == _claves_nuevas
assert not set(modelos_nuevos).intersection(modelos_originales)
print("Cinco clasificadores adicionales registrados, sin entrenar:", list(modelos_nuevos))


In [ ]:
# 4.4 — Interpretación de los resultados centralizados, sin entrenamiento adicional.
# Después de la evaluación integral de 4.6.1, usar desde una celda:
# interpretar_aporte_santander(tabla_validacion_ultima)
# Esta función solo lee la tabla recibida. No inicia otra comparación, reinicia
# experimentos ni modifica los registros o mecanismos de evaluación compartidos.

from IPython.display import display, Markdown

def interpretar_aporte_santander(tabla_cv):
    """Describe resultados observados de CV; nunca infiere cifras del holdout."""
    columnas = {"tipo", "modelo", "escalador", "cv_accuracy", "cv_precision",
                "cv_recall", "cv_f1", "cv_roc_auc", "cv_tiempo_s", "estado"}
    if not isinstance(tabla_cv, pd.DataFrame) or not columnas.issubset(tabla_cv.columns):
        raise ValueError("Se requiere la tabla CV publicada por ejecutar_comparacion en 4.6.1.")
    claves = ("svm_lineal", "adaboost", "knn", "lightgbm", "extra_trees")
    nuevas = tabla_cv[tabla_cv["tipo"].eq("nuevo") & tabla_cv["modelo"].isin(claves)].copy()
    if nuevas.empty:
        raise ValueError("No hay resultados de los cinco clasificadores adicionales.")
    display(nuevas.round(4))
    for nombre in claves:
        filas = nuevas[nuevas["modelo"].eq(nombre)]
        correctas = filas[filas["estado"].eq("OK")].dropna(subset=["cv_roc_auc"])
        if correctas.empty:
            display(Markdown(f"**{nombre}: PENDIENTE.** No hay folds completos con ROC-AUC válido."))
            continue
        # Mismo criterio de selección que Marco: AUC, Recall y F1 de CV.
        ordenadas = correctas.sort_values(
            ["cv_roc_auc", "cv_recall", "cv_f1"], ascending=False, kind="stable")
        mejor = ordenadas.iloc[0]
        empatados = ordenadas[np.isclose(ordenadas["cv_roc_auc"], mejor["cv_roc_auc"], rtol=0, atol=1e-12)]
        escaladores_auc = ", ".join(empatados["escalador"].astype(str))
        completo = (set(filas["escalador"]) == {"standard", "minmax", "robust", "quantile"}
                    and len(filas) == 4 and filas["estado"].eq("OK").all())
        display(Markdown(
            f"**{nombre}:** mayor ROC-AUC de CV con {escaladores_auc}; "
            f"configuración seleccionable por AUC/Recall/F1: {mejor['escalador']}. "
            f"Accuracy={mejor['cv_accuracy']:.4f}, Precision={mejor['cv_precision']:.4f}, "
            f"Recall={mejor['cv_recall']:.4f}, F1={mejor['cv_f1']:.4f}, "
            f"ROC-AUC={mejor['cv_roc_auc']:.4f}; ajuste medio={mejor['cv_tiempo_s']:.2f} s. "
            + ("Se compararon los cuatro escaladores." if completo else
               "Comparación parcial: faltan escaladores o existen errores; no concluir superioridad global.")))

    # Comparaciones requeridas: AdaBoost/DT, LightGBM/XGBoost/CatBoost, Extra Trees/RF.
    referencias = {
        "adaboost": ("decision_tree",),
        "lightgbm": ("xgboost", "catboost"),
        "extra_trees": ("random_forest",),
    }
    for nuevo, originales in referencias.items():
        for original in originales:
            izquierda = nuevas[nuevas["modelo"].eq(nuevo) & nuevas["estado"].eq("OK")]
            derecha = tabla_cv[tabla_cv["tipo"].eq("original")
                & tabla_cv["modelo"].eq(original) & tabla_cv["estado"].eq("OK")]
            pares = izquierda.merge(derecha, on="escalador", suffixes=("_nuevo", "_original"))
            if pares.empty:
                print(f"PENDIENTE comparación {nuevo}/{original}: faltan resultados comparables.")
                continue
            comparacion = pares[["escalador"]].copy()
            for metrica in ("accuracy", "precision", "recall", "f1", "roc_auc", "tiempo_s"):
                comparacion[f"delta_{metrica}"] = (
                    pares[f"cv_{metrica}_nuevo"] - pares[f"cv_{metrica}_original"])
            print(f"{nuevo} menos {original}, por el mismo escalador y protocolo CV:")
            display(comparacion.round(4))
    display(Markdown(
        "**Interpretación:** en SVM el escalamiento modifica la geometría del margen; "
        "en KNN modifica las distancias. Los ensambles de árboles pueden ser menos "
        "sensibles a cambios de escala, pero se contrasta en las tablas. Un delta positivo "
        "favorece al nuevo modelo en las métricas; en tiempo indica mayor costo de ajuste. "
        "Los tiempos excluyen predicción, por lo que no describen todo el costo de KNN. "
        "Las diferencias de medias no prueban significancia estadística. Los modelos "
        "originales pueden usar pesos de clase distintos; la comparación corresponde a "
        "las configuraciones completas. No se cambian umbrales con el holdout."))


## 4.5. Cinco modelos híbridos (ensambles)

Cada híbrido debe construirse con estimadores presentes en `modelos_originales` o `modelos_nuevos`. Registrar **H1–H5** en `modelos_hibridos`. En Voting Soft solo usar clasificadores que entregan probabilidades; en Stacking prevenir fugas con CV sobre entrenamiento. Todos se evalúan mediante la interfaz compartida. Los híbridos se registran por separado de los 11 modelos individuales.

### 4.5.1. H1 — VotingClassifier Hard: Logistic Regression + Decision Tree + KNN

> **Responsable: Brickzon**

**Implementar:** `VotingClassifier(voting='hard')` con los tres estimadores indicados; registrar clave `H1_voting_hard`. Incorporar preprocesamiento adecuado para KNN/Logistic Regression dentro del procedimiento acordado.

**Explicar:** la etiqueta final se determina por voto mayoritario. **Precaución de evaluación:** en general hard voting no expone probabilidades; **no inventar ROC-AUC**. Si no existe puntuación apropiada, registrar `NaN` para ROC-AUC o evaluar una variante soft aparte documentada sin sustituir H1.

**Entregable:** híbrido entrenable, métricas válidas y breve explicación del mecanismo de voto.

In [ ]:
# 4.5.1 H1 — VotingClassifier Hard: Logistic Regression + Decision Tree + KNN
from sklearn.ensemble import VotingClassifier
from sklearn.base import clone

if "modelos_hibridos" not in globals():
    modelos_hibridos = {}

# Validación estricta de diccionarios compartidos
if "modelos_originales" not in globals():
    raise NameError("El diccionario 'modelos_originales' (responsabilidad de Marco) no está definido en el entorno.")
if "modelos_nuevos" not in globals():
    raise NameError("El diccionario 'modelos_nuevos' (responsabilidad de Aldana) no está definido en el entorno.")

claves_requeridas_orig = ["logistic_regression", "decision_tree"]
for clave in claves_requeridas_orig:
    if clave not in modelos_originales:
        raise KeyError(f"Falta el modelo '{clave}' en 'modelos_originales'. Debe registrarse previamente en 4.3.")

if "knn" not in modelos_nuevos:
    raise KeyError("Falta el modelo 'knn' en 'modelos_nuevos'. Debe registrarse previamente en 4.4.3.")

# Construcción exclusiva a partir de los estimadores registrados, sin sustitución alternativa
est_lr = clone(modelos_originales["logistic_regression"])
est_dt = clone(modelos_originales["decision_tree"])
est_knn = clone(modelos_nuevos["knn"])

h1_voting_hard = VotingClassifier(
    estimators=[
        ("logistic_regression", est_lr),
        ("decision_tree", est_dt),
        ("knn", est_knn)
    ],
    voting="hard"
)

modelos_hibridos["H1_voting_hard"] = h1_voting_hard

print("H1 registrado con éxito en modelos_hibridos['H1_voting_hard']:")
print(" - Estimadores: Logistic Regression, Decision Tree, KNN (obtenidos de modelos_originales y modelos_nuevos)")
print(" - Tipo de ensamble: VotingClassifier(voting='hard') con voto mayoritario")
print(" - Preprocesamiento: KNN y Logistic Regression son sensibles al escalado; el ensamble completo se evaluará mediante el pipeline común en 4.6.")
print(" - Métrica ROC-AUC: Al utilizar voting='hard' no se dispone de probabilidades continuas; se reportará NaN por protocolo en la evaluación centralizada.")


### 4.5.2. H2 — VotingClassifier Soft: Random Forest + AdaBoost + Logistic Regression

> **Responsable: Brickzon**

**Implementar:** `VotingClassifier(voting='soft')` con los tres estimadores; registrar `H2_voting_soft`.

**Explicar:** combina probabilidades estimadas de clase 1. Comprobar que los tres modelos tengan `predict_proba`; usar sus puntuaciones para ROC-AUC de forma correcta. Escalamiento ajustado solo en entrenamiento.

**Entregable:** híbrido entrenable, métricas y comparación con cada componente.

In [ ]:
# 4.5.2 H2 — VotingClassifier Soft: Random Forest + AdaBoost + Logistic Regression
from sklearn.ensemble import VotingClassifier
from sklearn.base import clone

if "modelos_hibridos" not in globals():
    modelos_hibridos = {}

# Validación estricta de diccionarios compartidos
if "modelos_originales" not in globals():
    raise NameError("El diccionario 'modelos_originales' (responsabilidad de Marco) no está definido en el entorno.")
if "modelos_nuevos" not in globals():
    raise NameError("El diccionario 'modelos_nuevos' (responsabilidad de Aldana) no está definido en el entorno.")

claves_requeridas_orig = ["random_forest", "logistic_regression"]
for clave in claves_requeridas_orig:
    if clave not in modelos_originales:
        raise KeyError(f"Falta el modelo '{clave}' en 'modelos_originales'. Debe registrarse previamente en 4.3.")

if "adaboost" not in modelos_nuevos:
    raise KeyError("Falta el modelo 'adaboost' en 'modelos_nuevos'. Debe registrarse previamente en 4.4.2.")

# Construcción exclusiva a partir de los estimadores registrados
est_rf = clone(modelos_originales["random_forest"])
est_ada = clone(modelos_nuevos["adaboost"])
est_lr = clone(modelos_originales["logistic_regression"])

# Comprobación previa de existencia del método predict_proba
estimadores_h2 = [
    ("random_forest", est_rf),
    ("adaboost", est_ada),
    ("logistic_regression", est_lr)
]

for nombre, est in estimadores_h2:
    if not hasattr(est, "predict_proba"):
        raise AttributeError(f"El clasificador '{nombre}' no dispone del método 'predict_proba', requerido para voting='soft'.")

h2_voting_soft = VotingClassifier(
    estimators=estimadores_h2,
    voting="soft"
)

modelos_hibridos["H2_voting_soft"] = h2_voting_soft

print("H2 registrado con éxito en modelos_hibridos['H2_voting_soft']:")
print(" - Estimadores: Random Forest, AdaBoost, Logistic Regression (obtenidos de modelos_originales y modelos_nuevos)")
print(" - Tipo de ensamble: VotingClassifier(voting='soft') con promedio de probabilidades")
print(" - Verificación predict_proba: Se comprobó que los tres clasificadores disponen de predict_proba para estimar las probabilidades de clase.")


### 4.5.3. H3 — VotingClassifier Soft: XGBoost + LightGBM + CatBoost

> **Responsable: Brickzon**

**Implementar:** `VotingClassifier(voting='soft')` con XGBoost, LightGBM y CatBoost; registrar `H3_voting_soft_boosting`.

**Precaución Colab:** usar versiones/argumentos compatibles, preferentemente CPU sin forzar GPU ni usar opciones obsoletas. Mantener semilla y criterios de evaluación comunes; usar estimadores clonables.

**Entregable:** híbrido entrenable, métricas y explicación de si combinar tres métodos boosting agrega valor respecto a cada uno.

In [ ]:
# 4.5.3 H3 — VotingClassifier Soft: XGBoost + LightGBM + CatBoost
from sklearn.ensemble import VotingClassifier
from sklearn.base import clone

if "modelos_hibridos" not in globals():
    modelos_hibridos = {}

# Validación estricta de diccionarios compartidos
if "modelos_originales" not in globals():
    raise NameError("El diccionario 'modelos_originales' (responsabilidad de Marco) no está definido en el entorno.")
if "modelos_nuevos" not in globals():
    raise NameError("El diccionario 'modelos_nuevos' (responsabilidad de Aldana) no está definido en el entorno.")

claves_requeridas_orig = ["xgboost", "catboost"]
for clave in claves_requeridas_orig:
    if clave not in modelos_originales:
        raise KeyError(f"Falta el modelo '{clave}' en 'modelos_originales'. Debe registrarse previamente en 4.3.")

if "lightgbm" not in modelos_nuevos:
    raise KeyError("Falta el modelo 'lightgbm' en 'modelos_nuevos'. Debe registrarse previamente en 4.4.4.")

# Construcción exclusiva a partir de los estimadores registrados
est_xgb = clone(modelos_originales["xgboost"])
est_lgb = clone(modelos_nuevos["lightgbm"])
est_cat = clone(modelos_originales["catboost"])

# Garantizar compatibilidad con Colab CPU informando cualquier ajuste de forma explícita
# 1. XGBoost: normalizar variantes de GPU (cuda, gpu, cuda:0, gpu:0, etc.) a CPU y tree_method a 'hist'
xgb_params = est_xgb.get_params()
ajustes_xgb = {}
tm = str(xgb_params.get("tree_method") or "").lower()
dev = str(xgb_params.get("device") or "").lower()

if dev in ["cuda", "gpu", "cuda:0", "gpu:0"] or any(dev.startswith(p) for p in ["cuda", "gpu"]):
    ajustes_xgb["device"] = "cpu"

if tm in ["gpu_hist", "cuda"] or "gpu" in tm:
    ajustes_xgb["tree_method"] = "hist"
    if "device" in xgb_params:
        ajustes_xgb["device"] = "cpu"

if ajustes_xgb:
    print(f"[Aviso] Ajuste explícito en XGBoost para CPU de Colab: {ajustes_xgb}")
    est_xgb.set_params(**ajustes_xgb)

# 2. LightGBM: normalizar a device_type='cpu' evitando valores contradictorios con alias device
lgb_params = est_lgb.get_params()
ajustes_lgb = {}
dev_type = str(lgb_params.get("device_type") or "").lower()
dev_alias = str(lgb_params.get("device") or "").lower()

if dev_type in ["gpu", "cuda"] or dev_alias in ["gpu", "cuda"] or any(d.startswith(("gpu", "cuda")) for d in [dev_type, dev_alias] if d):
    ajustes_lgb["device_type"] = "cpu"
    if "device" in lgb_params and lgb_params["device"] is not None:
        ajustes_lgb["device"] = "cpu"

if ajustes_lgb:
    print(f"[Aviso] Ajuste explícito en LightGBM para CPU de Colab: {ajustes_lgb}")
    est_lgb.set_params(**ajustes_lgb)

# 3. CatBoost: verificar task_type
cat_params = est_cat.get_params()
ajustes_cat = {}
task_type = str(cat_params.get("task_type") or "").upper()
if task_type in ["GPU", "CUDA"] or "GPU" in task_type:
    ajustes_cat["task_type"] = "CPU"
if ajustes_cat:
    print(f"[Aviso] Ajuste explícito en CatBoost para CPU de Colab: {ajustes_cat}")
    est_cat.set_params(**ajustes_cat)

# Comprobación previa de existencia del método predict_proba
estimadores_h3 = [
    ("xgboost", est_xgb),
    ("lightgbm", est_lgb),
    ("catboost", est_cat)
]

for nombre, est in estimadores_h3:
    if not hasattr(est, "predict_proba"):
        raise AttributeError(f"El clasificador '{nombre}' no dispone del método 'predict_proba', requerido para voting='soft'.")

h3_voting_soft_boosting = VotingClassifier(
    estimators=estimadores_h3,
    voting="soft"
)

modelos_hibridos["H3_voting_soft_boosting"] = h3_voting_soft_boosting

print("H3 registrado con éxito en modelos_hibridos['H3_voting_soft_boosting']:")
print(" - Estimadores: XGBoost, LightGBM, CatBoost (obtenidos de modelos_originales y modelos_nuevos)")
print(" - Tipo de ensamble: VotingClassifier(voting='soft')")
print(" - Configuración de ejecución: Compatibilidad CPU verificada para Colab.")
print(f"Total de modelos híbridos registrados en modelos_hibridos: {len(modelos_hibridos)}")


### 4.5.4. H4 — StackingClassifier: SVM + KNN + Random Forest

> **Responsable: Vera**

**Implementar:** `StackingClassifier` con `LinearSVC`, `KNeighborsClassifier` y `RandomForestClassifier` como modelos base; `LogisticRegression` como meta-modelo. Registrar `H4_stacking_svm`.

**Condiciones:** emplear `cv=3` (o justificar otro valor), `stack_method='auto'` y preprocesamiento compatible; construir pipelines internos para modelos sensibles a escala cuando corresponda. El meta-modelo se entrena con predicciones obtenidas **mediante CV del entrenamiento**; nunca con etiquetas del holdout.

**Entregable:** híbrido, reporte de costo de entrenamiento, métricas y explicación de las dos capas.

### 4.5.5. H5 — StackingClassifier: Decision Tree + Extra Trees + AdaBoost

> **Responsable: Vera**

**Implementar:** `StackingClassifier` con `DecisionTreeClassifier`, `ExtraTreesClassifier` y `AdaBoostClassifier` como modelos base; `LogisticRegression` como meta-modelo. Registrar `H5_stacking_arboles`.

**Condiciones:** CV interna sobre entrenamiento, semillas consistentes, evitar remuestreo fuera de folds y no modificar datos de validación/holdout. Medir tiempo y observar sobreajuste.

**Entregable:** híbrido, métricas comparables, contraste con H4 y comprobación de `len(modelos_hibridos) == 5`.

## 4.6. Evaluación final, auditoría e integración

### 4.6.1. Consolidar resultados y seleccionar configuraciones mediante validación

> **Responsable: Marco**

**Implementar:** ejecutar los clasificadores `modelos_originales` y `modelos_nuevos` bajo los cuatro escaladores usando la misma estrategia de validación (idealmente CV estratificada en **entrenamiento**). Para los híbridos, seleccionar escalador/preprocesamiento usando la misma validación, registrando la decisión. Consolidar `resultados` en `tabla_resultados` con columnas homogéneas.

**Matriz sugerida:** hasta **4 × 11 = 44 configuraciones individuales** y **5 híbridos**. Esto es una **propuesta de evaluación**, no un número expresamente exigido por el profesor; si el costo de Colab obliga a priorizar, documentar cobertura real y justificarla. Si se usa una muestra exploratoria, las comparaciones deben usar la **misma muestra** y las conclusiones finales deben identificarse como tales.

**Selección:** priorizar ROC-AUC, Recall y F1 de clase 1; Accuracy no debe decidir sola por el desbalanceo. No elegir repetidamente configuraciones mirando el holdout.

**Entregable:** tabla completa, ordenamiento por métrica prioritaria y registro de configuraciones efectivamente probadas.

In [ ]:
# 4.6.1 — CV solo en entrenamiento; el holdout se evalúa por separado y a petición explícita.
import warnings
import copy

COLUMNAS_CV = ["tipo", "modelo", "escalador", "cv_accuracy", "cv_precision", "cv_recall",
    "cv_f1", "cv_roc_auc", "cv_tiempo_s", "folds", "estado"]
_estado_anterior = globals().get("ESTADO_COMPARACION")
if isinstance(_estado_anterior, dict):
    ESTADO_COMPARACION = _estado_anterior
    ESTADO_COMPARACION.setdefault("revision", 0)
    ESTADO_COMPARACION.setdefault("validacion", pd.DataFrame())
    ESTADO_COMPARACION.setdefault("modelos", {})
    ESTADO_COMPARACION.setdefault("escaladores", {})
    ESTADO_COMPARACION.setdefault("holdout_status", "sin_experimento")
    ESTADO_COMPARACION.setdefault("evaluada_revision", None)
    ESTADO_COMPARACION.setdefault("seleccion_holdout_evaluada", None)
else:
    ESTADO_COMPARACION = {"revision": 0, "validacion": pd.DataFrame(), "modelos": {},
        "escaladores": {}, "holdout_status": "sin_experimento",
        "evaluada_revision": None, "seleccion_holdout_evaluada": None}

def _obtener_diccionario(nombre):
    value = globals().get(nombre)
    return value if isinstance(value, dict) else {}

def _media_segura(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(values.mean()) if len(values) else np.nan

def _publicar_tablas(tabla_validacion=None):
    """Publica las tablas compartidas, sin asignaciones manuales en las celdas llamadoras."""
    if tabla_validacion is not None:
        globals()["tabla_validacion_ultima"] = tabla_validacion.copy(deep=True)
    _publicar_resultados()
    return globals()["tabla_resultados"], globals().get("tabla_validacion_ultima", pd.DataFrame())

def ejecutar_comparacion(*, modelos=None, escaladores_a_usar=None, incluir_hibridos=True,
                          modo="rapido", n_splits=3, limite_modelos_rapido=3,
                          exportar_csv=None, reiniciar_experimento=False):
    """Recalcula CV estratificada desde cero sobre X_train. No evalúa ni selecciona con métricas del holdout; la validación solo verifica que los índices train/holdout sean disjuntos.

    No usa caché CV. Tras iniciar una selección, repetir exige reiniciar_experimento=True,
    que descarta explícitamente la selección/holdout anteriores.
    """
    if modo not in {"rapido", "completo"}:
        raise ValueError("modo debe ser 'rapido' o 'completo'.")
    if ESTADO_COMPARACION.get("holdout_status") == "evaluando":
        raise RuntimeError("No se puede reiniciar mientras la evaluación holdout está en curso.")
    if ESTADO_COMPARACION.get("revision", 0) > 0 and not reiniciar_experimento:
        raise RuntimeError("Ya existe una selección CV. Usa reiniciar_experimento=True para descartarla y recalcular.")
    if not isinstance(globals().get("X_train"), pd.DataFrame) or not isinstance(globals().get("y_train"), pd.Series):
        raise RuntimeError("Falta el split de Poma: X_train/y_train deben ser DataFrame/Series.")
    if n_splits < 2 or y_train.value_counts().min() < n_splits:
        raise ValueError("Cada clase de y_train debe contener al menos n_splits observaciones.")

    # Una nueva selección invalida holdout y resultados de la exploración anterior.
    resultados.clear()
    modelos_holdout.clear()
    modelos_validacion.clear()
    predicciones_holdout.clear()
    revision = ESTADO_COMPARACION.get("revision", 0) + 1
    ESTADO_COMPARACION.update({"revision": revision, "validacion": pd.DataFrame(), "modelos": {},
        "escaladores": {}, "holdout_status": "seleccionando",
        "evaluada_revision": None, "seleccion_holdout_evaluada": None})
    _publicar_tablas(pd.DataFrame(columns=COLUMNAS_CV))

    grupos = [("original", _obtener_diccionario("modelos_originales")),
              ("nuevo", _obtener_diccionario("modelos_nuevos"))]
    if incluir_hibridos:
        grupos.append(("hibrido", _obtener_diccionario("modelos_hibridos")))
    disponibles = [(kind, name, model) for kind, group in grupos
                   for name, model in group.items() if model is not None]
    if modelos is not None:
        allowed = set(modelos)
        disponibles = [item for item in disponibles if item[1] in allowed or f"{item[0]}:{item[1]}" in allowed]
    if modo == "rapido" and modelos is None:
        disponibles = [item for kind in ("original", "nuevo", "hibrido")
                       for item in [x for x in disponibles if x[0] == kind][:limite_modelos_rapido]]
    if not disponibles:
        ESTADO_COMPARACION["holdout_status"] = "sin_modelos"
        print("PENDIENTE: los diccionarios de modelos aún no contienen estimadores.")
        return _publicar_tablas(pd.DataFrame(columns=COLUMNAS_CV))[1]

    scaler_dict = _obtener_diccionario("escaladores")
    scalers = [(name, scaler_dict[name]) for name in (escaladores_a_usar or list(scaler_dict))
               if name in scaler_dict]
    if not scalers:
        scalers = [("sin_escalado", None)]
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    cv_rows, model_snapshots, scaler_snapshots = [], {}, {}
    for kind, name, estimator in disponibles:
        scaler_options = scalers if kind != "hibrido" else [("sin_escalado", None)]
        try:
            model_snapshots[(kind, name)] = clone(estimator)
        except Exception:
            model_snapshots[(kind, name)] = copy.deepcopy(estimator)
        for scaler_name, scaler in scaler_options:
            try:
                scaler_snapshots[scaler_name] = clone(scaler) if scaler is not None else None
            except Exception:
                scaler_snapshots[scaler_name] = copy.deepcopy(scaler)
            fold_rows = []
            try:
                for fold, (fit_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
                    fold_rows.append(evaluar_modelo(name, kind, estimator, scaler,
                        X_fit=X_train.iloc[fit_idx], y_fit=y_train.iloc[fit_idx],
                        X_eval=X_train.iloc[val_idx], y_eval=y_train.iloc[val_idx],
                        etapa="validacion", guardar=False, fold=fold, nombre_escalador=scaler_name))
            except Exception as exc:
                cv_rows.append({"tipo": kind, "modelo": name, "escalador": scaler_name,
                    "cv_accuracy": np.nan, "cv_precision": np.nan, "cv_recall": np.nan,
                    "cv_f1": np.nan, "cv_roc_auc": np.nan, "cv_tiempo_s": np.nan,
                    "folds": len(fold_rows), "estado": f"ERROR: {type(exc).__name__}: {exc}"})
                continue
            for row in fold_rows:
                row["escalador"] = scaler_name
                resultados.append(row)
            cv_rows.append({"tipo": kind, "modelo": name, "escalador": scaler_name,
                "cv_accuracy": _media_segura([r["accuracy"] for r in fold_rows]),
                "cv_precision": _media_segura([r["precision"] for r in fold_rows]),
                "cv_recall": _media_segura([r["recall"] for r in fold_rows]),
                "cv_f1": _media_segura([r["f1"] for r in fold_rows]),
                "cv_roc_auc": _media_segura([r["roc_auc"] for r in fold_rows]),
                "cv_tiempo_s": _media_segura([r["tiempo_s"] for r in fold_rows]),
                "folds": len(fold_rows), "estado": "OK"})
    table_cv = pd.DataFrame(cv_rows, columns=COLUMNAS_CV)
    ESTADO_COMPARACION.update({"validacion": table_cv.copy(deep=True), "modelos": model_snapshots,
        "escaladores": scaler_snapshots, "holdout_status": "cv_disponible"})
    _publicar_tablas(table_cv)
    if exportar_csv:
        destination = Path(exportar_csv)
        destination.parent.mkdir(parents=True, exist_ok=True)
        table_cv.to_csv(destination.with_name(destination.stem + "_validacion.csv"), index=False)
        globals()["tabla_resultados"].to_csv(destination, index=False)
    print(f"CV terminada: {len(table_cv)} configuraciones; no se consultó el holdout.")
    print("Modelos disponibles:", {kind: sum(x[0] == kind for x in disponibles)
                                  for kind in ("original", "nuevo", "hibrido")})
    return table_cv


def evaluar_holdout_final():
    """Consume la selección CV vigente y autoriza una sola evaluación final."""
    state = ESTADO_COMPARACION
    table_cv = state.get("validacion")
    revision = state.get("revision")
    if not isinstance(table_cv, pd.DataFrame) or table_cv.empty or state.get("holdout_status") != "cv_disponible":
        raise RuntimeError("Ejecuta primero una CV nueva; no hay selección vigente pendiente de holdout.")
    if state.get("evaluada_revision") == revision:
        raise RuntimeError("Esta revisión CV ya se usó para evaluar holdout.")
    if not all(name in globals() for name in ("X_train", "y_train", "X_test", "y_test")):
        raise RuntimeError("Falta X_test/y_test para la evaluación final.")
    successful = table_cv[table_cv["estado"] == "OK"].copy()
    if successful.empty:
        raise RuntimeError("La validación no produjo configuraciones seleccionables.")
    successful["_roc_rank"] = successful["cv_roc_auc"].fillna(-np.inf)
    selected = successful.sort_values(["_roc_rank", "cv_recall", "cv_f1"], ascending=False)
    selected = selected.drop_duplicates(["tipo", "modelo"], keep="first")
    state["evaluada_revision"] = revision
    state["seleccion_holdout_evaluada"] = {
        "revision": revision,
        "configuraciones": selected[["tipo", "modelo", "escalador", "cv_roc_auc",
                                     "cv_recall", "cv_f1"]].to_dict("records"),
    }
    state["holdout_status"] = "evaluando"
    globals()["_HOLDOUT_ACTIVE_TOKEN"] = _HOLDOUT_GATE
    try:
        for _, row in selected.iterrows():
            kind, name, scaler_name = row["tipo"], row["modelo"], row["escalador"]
            estimator = state["modelos"].get((kind, name))
            scaler = state["escaladores"].get(scaler_name)
            if estimator is None:
                print(f"PENDIENTE holdout: falta snapshot de {kind}/{name}.")
                continue
            evaluar_modelo(name, kind, estimator, scaler, X_fit=X_train, y_fit=y_train,
                X_eval=X_test, y_eval=y_test, etapa="holdout", guardar=True,
                guardar_ajustado=True, calcular_matriz=True, nombre_escalador=scaler_name)
    finally:
        globals()["_HOLDOUT_ACTIVE_TOKEN"] = None
        state["holdout_status"] = "evaluado"
    _publicar_tablas()
    return globals()["tabla_resultados"].query("etapa == 'holdout'").copy()

print("Comparador definido. Repetir CV requiere reiniciar_experimento=True; el holdout es un paso separado.")

### 4.6.2. Verificar los cinco híbridos y sus métricas

> **Responsable: Vera**

**Implementar:** comprobar que `modelos_hibridos` contiene H1–H5 y que sus componentes están presentes entre los modelos individuales. Evaluar los cinco con la interfaz común y un preprocesamiento comparable.

**Auditoría mínima:** comparar H1/H2/H3 frente a sus componentes; H4/H5 frente a sus modelos base; revisar `ROC-AUC` con etiquetas reales y puntuaciones (para hard voting, `NaN` si no hay puntuación válida); registrar tiempos y errores/convergencia sin ocultarlos.

**Entregable:** tabla `tabla_hibridos`, con cinco filas por la configuración final elegida y una nota sobre rendimiento/costo.

### 4.6.3. Generar visualizaciones y matrices de confusión

> **Responsable: Marco**

**Implementar:** tabla ordenada de rendimiento individual e híbrido, gráficos legibles (por ejemplo, barras de ROC-AUC/F1/Recall) y matrices de confusión de los modelos más relevantes usando el **holdout final**. Si se trazan curvas ROC, usar predicciones probabilísticas o puntuaciones sobre `X_test` y `y_test`, sin invertir los argumentos.

**Entregable:** gráficos con etiquetas y títulos, interpretación de falsos negativos (clientes insatisfechos no detectados) y advertencia del desbalanceo.

**Verificación:** las etiquetas y los cálculos provienen de la misma partición, no mezclan validación con holdout.

In [ ]:
# 4.6.3 — Las gráficas leen las tablas publicadas por la interfaz común.
def _tabla_graficable(tabla=None, etapa="holdout"):
    if etapa not in {"validacion", "holdout"}:
        raise ValueError("etapa debe ser 'validacion' o 'holdout'.")
    if tabla is None:
        tabla = globals().get("tabla_validacion_ultima") if etapa == "validacion" else globals().get("tabla_resultados")
    if not isinstance(tabla, pd.DataFrame) or tabla.empty:
        print(f"PENDIENTE: no hay tabla consolidada de {etapa}.")
        return pd.DataFrame()
    datos = tabla.copy()
    if etapa == "validacion":
        mapping = {"accuracy": "cv_accuracy", "precision": "cv_precision", "recall": "cv_recall",
                   "f1": "cv_f1", "roc_auc": "cv_roc_auc", "tiempo_s": "cv_tiempo_s"}
        if "cv_accuracy" not in datos:
            print("PENDIENTE: tabla_validacion_ultima no contiene métricas CV.")
            return pd.DataFrame()
        for target, source in mapping.items():
            datos[target] = datos[source]
    elif "etapa" in datos.columns:
        datos = datos[datos["etapa"] == "holdout"]
    return datos.dropna(subset=["modelo"]).copy()

def mostrar_tabla_resultados(tabla=None, etapa="holdout", ordenar_por=None):
    datos = _tabla_graficable(tabla, etapa)
    if datos.empty:
        return datos
    sort_key = ordenar_por or ("roc_auc" if etapa == "holdout" else "cv_roc_auc")
    if sort_key in datos.columns:
        datos = datos.sort_values(sort_key, ascending=False, na_position="last")
    columns = [c for c in (COLUMNAS_CV if etapa == "validacion" else COLUMNAS_RESULTADOS)
               if c in datos.columns]
    display(datos[columns].reset_index(drop=True))
    return datos

def graficar_metricas(tabla=None, etapa="holdout"):
    datos = _tabla_graficable(tabla, etapa)
    if datos.empty:
        return
    datos["configuracion"] = datos["tipo"].astype(str) + " · " + datos["modelo"].astype(str) + " · " + datos["escalador"].astype(str)
    label = "CV media (selección)" if etapa == "validacion" else "Holdout final"
    for metric, title in [("accuracy", "Accuracy"), ("roc_auc", "ROC-AUC clase 1"),
                          ("tiempo_s", "Tiempo de entrenamiento medio (s)")]:
        if metric not in datos or datos[metric].notna().sum() == 0:
            continue
        frame = datos.sort_values(metric, ascending=(metric == "tiempo_s"), na_position="last")
        fig, ax = plt.subplots(figsize=(max(9, min(18, 0.65 * len(frame))), 5))
        sns.barplot(data=frame, x="configuracion", y=metric, hue="tipo", dodge=False, ax=ax)
        ax.set(title=f"{title} — {label}", xlabel="Configuración", ylabel=title)
        ax.tick_params(axis="x", rotation=70)
        ax.legend(title="Tipo", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout(); plt.show()
    metrics = [m for m in ("precision", "recall", "f1") if m in datos and datos[m].notna().any()]
    if metrics:
        melted = datos.melt(id_vars=["tipo", "modelo", "escalador"], value_vars=metrics,
                            var_name="métrica", value_name="valor")
        melted["configuracion"] = melted["tipo"].astype(str) + " · " + melted["modelo"].astype(str) + " · " + melted["escalador"].astype(str)
        fig, ax = plt.subplots(figsize=(max(10, min(18, 0.7 * len(datos))), 6))
        sns.barplot(data=melted, x="configuracion", y="valor", hue="métrica", ax=ax)
        ax.set(title=f"Precision, Recall y F1 — {label}", xlabel="Configuración", ylabel="Puntaje")
        ax.tick_params(axis="x", rotation=70)
        ax.legend(title="Métrica", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout(); plt.show()

def graficar_matrices_confusion(max_matrices=6):
    datos = _tabla_graficable(etapa="holdout")
    if datos.empty:
        return
    datos = datos[datos["matriz_confusion"].notna()].head(max_matrices)
    if datos.empty:
        print("PENDIENTE: no hay matrices del holdout final.")
        return
    for _, row in datos.iterrows():
        cm = np.asarray(row["matriz_confusion"])
        fig, ax = plt.subplots(figsize=(4.5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Pred. satisfecho (0)", "Pred. insatisfecho (1)"],
            yticklabels=["Real satisfecho (0)", "Real insatisfecho (1)"])
        ax.set(title=f"Matriz de confusión: {row['modelo']} ({row['tipo']})",
               xlabel="Predicción", ylabel="Valor real")
        plt.tight_layout(); plt.show()

def graficar_curvas_roc(max_curvas=8):
    pairs = list(predicciones_holdout.items())
    if not pairs:
        print("PENDIENTE: no hay scores continuos de holdout final para dibujar ROC.")
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    for (kind, name, scaler_name), values in pairs[:max_curvas]:
        if values.get("score") is None:
            continue
        fpr, tpr, _ = roc_curve(values["y_true"], values["score"])
        auc = roc_auc_score(values["y_true"], values["score"])
        ax.plot(fpr, tpr, label=f"{name} [{scaler_name}] AUC={auc:.3f}")
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Azar")
    ax.set(title="Curvas ROC — holdout final", xlabel="Tasa de falsos positivos",
           ylabel="Recall / tasa de verdaderos positivos")
    ax.legend(loc="lower right", fontsize="small"); plt.tight_layout(); plt.show()

def graficar_individuales_vs_hibridos(etapa="validacion", metrica="roc_auc"):
    datos = _tabla_graficable(etapa=etapa)
    if datos.empty:
        return
    datos = datos[datos["tipo"].isin(["original", "nuevo", "hibrido"])]
    if metrica not in datos or not datos[metrica].notna().any():
        print(f"PENDIENTE: no hay valores observados de {metrica} en {etapa}.")
        return
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(data=datos, x="tipo", y=metrica, ax=ax)
    sns.stripplot(data=datos, x="tipo", y=metrica, color="black", alpha=.65, ax=ax)
    label = "CV para selección" if etapa == "validacion" else "holdout final"
    ax.set(title=f"Individuales e híbridos: {metrica} ({label})",
           xlabel="Familia de modelo", ylabel=metrica)
    plt.tight_layout(); plt.show()

print("Visualizaciones conectadas a tabla_validacion_ultima (CV) y tabla_resultados (holdout).")

### 4.6.4. Redactar interpretación, conclusiones y limitaciones

> **Responsable: Marco**

**Implementar en celdas de texto Markdown debajo de este bloque:** justificar cuál escalador funcionó mejor y para qué modelos; comparar individuales versus híbridos; explicar qué métricas sostienen la selección; comentar tiempo computacional, desbalanceo y límites de la evaluación.

**No inventar resultados.** Completar las conclusiones únicamente después de ejecutar las celdas y revisar la tabla consolidada. Indicar si alguna combinación se omitió y el motivo.

**Entregable:** un párrafo de resultados, un párrafo comparativo y un párrafo de limitaciones y mejoras. La celda de código vacía se deja como espacio opcional para cálculos auxiliares de este análisis.

In [ ]:
# 4.6.4 — Conclusiones: selección por CV y holdout exclusivamente como evaluación final.
def preparar_resumen_conclusiones():
    cv = globals().get("tabla_validacion_ultima")
    if not isinstance(cv, pd.DataFrame) or cv.empty:
        print("PENDIENTE: no hay métricas CV observadas; ejecuta ejecutar_comparacion().")
        return None
    cv = cv[cv["estado"] == "OK"].copy()
    if cv.empty:
        print("PENDIENTE: no hay configuraciones CV exitosas.")
        return None
    cv["_roc_rank"] = cv["cv_roc_auc"].fillna(-np.inf)
    ranked = cv.sort_values(["_roc_rank", "cv_recall", "cv_f1"], ascending=False)
    selected = ranked.drop_duplicates(["tipo", "modelo"], keep="first")
    cv_columns = ["tipo", "modelo", "escalador", "cv_roc_auc", "cv_recall",
                  "cv_f1", "cv_accuracy", "cv_tiempo_s"]
    print("Selección: ranking basado exclusivamente en CV sobre entrenamiento (ROC-AUC, Recall y F1).")
    display(selected[[c for c in cv_columns if c in selected]].reset_index(drop=True))
    full = globals().get("tabla_resultados")
    if not isinstance(full, pd.DataFrame) or full.empty or not (full["etapa"] == "holdout").any():
        print("Holdout: pendiente; la selección se informa con CV, sin evaluación final todavía.")
        return selected
    final = full[full["etapa"] == "holdout"]
    keys = ["tipo", "modelo", "escalador"]
    final_selected = selected[keys].merge(final, on=keys, how="left", validate="one_to_one")
    print("Holdout: resultados finales de las configuraciones elegidas por CV; no altera el ranking.")
    columns = keys + ["roc_auc", "recall", "f1", "accuracy", "tiempo_s"]
    display(final_selected[[c for c in columns if c in final_selected]].reset_index(drop=True))
    print("Interpreta falsos negativos con Recall/F1 y el desbalanceo. Completa limitaciones solo con observaciones reales.")
    return selected

#### Plantilla para completar tras ejecutar la evaluación

- **Escaladores:** compara las métricas medias de validación (ROC-AUC, Recall y F1) de los cuatro transformadores por modelo. Describe qué familias se benefician; no extrapoles al holdout configuraciones no seleccionadas.
- **Modelos originales y nuevos:** informa el ranking validado y luego las métricas holdout de las configuraciones seleccionadas. Debido al desbalanceo, interpreta los falsos negativos junto con Recall, F1 y ROC-AUC de la clase 1.
- **Híbridos y mejor modelo:** compara cada híbrido con sus componentes y reporta la configuración elegida por validación, su resultado final y tiempo. No selecciones retrospectivamente por el holdout.
- **Limitaciones y mejoras:** registra combinaciones omitidas, duración, errores de convergencia, variabilidad entre folds y la limitación de una sola partición holdout. Completa estos puntos solo con resultados observados.


### 4.6.5. Control final de integración y ejecución en Google Colab

> **Responsable: Marco**

**Implementar:** comprobaciones automáticas/focalizadas que verifiquen: 4 escaladores, 6 modelos originales, 5 modelos nuevos, 5 híbridos; forma y clases de `X_train/X_test/y_train/y_test`; inexistencia de `ID` entre predictores; ausencia de fuga antes de split; nombres únicos de modelos y métricas válidas. Verificar que `tabla_resultados` no esté vacía y no contenga resultados inventados.

**Entrega:** guardar el `.ipynb` final con Markdown completo, celdas de ejecución y resultados reales, más una copia descargable desde Colab. Si se utilizó Drive, eliminar dependencias de rutas privadas o documentar cómo reconstruirlas. Ejecutar desde una sesión limpia **la Parte B en orden**, sin depender de variables definidas en la Parte A.

**Criterio de aceptación:** cualquier integrante puede abrir el notebook, ubicar los datos, seguir las instrucciones, ejecutar el bloque grupal y entender quién hizo cada parte.

In [ ]:
# 4.6.5 — Auditoría estructural y funcional; fixtures técnicos no son resultados científicos.

def _validar_esquema_resultados_impl(tabla):
    required = {"tipo", "modelo", "escalador", "accuracy", "precision", "recall",
                "f1", "roc_auc", "tiempo_s", "etapa"}
    if not isinstance(tabla, pd.DataFrame):
        return "FAIL", "tabla_resultados no es DataFrame."
    missing = required - set(tabla.columns)
    if missing:
        return "FAIL", f"Faltan columnas: {sorted(missing)}."
    if tabla.empty:
        return "PENDIENTE", "La tabla no contiene resultados observados."
    if not tabla["tipo"].isin(["original", "nuevo", "hibrido"]).all():
        return "FAIL", "tipo tiene valores ajenos al contrato."
    if not tabla["etapa"].isin(["validacion", "holdout"]).all():
        return "FAIL", "etapa tiene valores ajenos al contrato."
    for col in ["accuracy", "precision", "recall", "f1"]:
        values = pd.to_numeric(tabla[col], errors="coerce")
        if values.isna().any() or not values.between(0, 1).all():
            return "FAIL", f"Métrica {col} faltante, no numérica o fuera de [0,1]."
    auc = pd.to_numeric(tabla["roc_auc"], errors="coerce")
    if (tabla["roc_auc"].notna() & auc.isna()).any() or not auc.dropna().between(0, 1).all():
        return "FAIL", "roc_auc debe estar en [0,1] o NaN si no hay score continuo."
    elapsed = pd.to_numeric(tabla["tiempo_s"], errors="coerce")
    if elapsed.isna().any() or (elapsed < 0).any():
        return "FAIL", "tiempo_s debe ser numérico y no negativo."
    return "PASS", f"{len(tabla)} filas con esquema y métricas válidos."

def _validar_esquema_cv_impl(tabla):
    if not isinstance(tabla, pd.DataFrame):
        return "FAIL", "tabla_validacion_ultima no es DataFrame."
    missing = set(COLUMNAS_CV) - set(tabla.columns)
    if missing:
        return "FAIL", f"Faltan columnas CV: {sorted(missing)}."
    if tabla.empty:
        return "PENDIENTE", "Aún no hay configuraciones CV."
    successful = tabla[tabla["estado"] == "OK"]
    for col in ["cv_accuracy", "cv_precision", "cv_recall", "cv_f1"]:
        values = pd.to_numeric(successful[col], errors="coerce")
        if values.isna().any() or not values.between(0, 1).all():
            return "FAIL", f"{col} de filas OK debe estar en [0,1]."
    return "PASS", f"{len(tabla)} configuraciones CV con esquema válido."

def _validar_esquema_resultados(tabla):
    try:
        return _validar_esquema_resultados_impl(tabla)
    except Exception as exc:
        return "FAIL", f"Error al comprobar esquema de métricas: {type(exc).__name__}: {exc}"

def _validar_esquema_cv(tabla):
    try:
        return _validar_esquema_cv_impl(tabla)
    except Exception as exc:
        return "FAIL", f"Error al comprobar esquema CV: {type(exc).__name__}: {exc}"

def _prueba_funcional_pipeline_y_holdout():
    """Prueba interfaz, CV, cierre del holdout y ajuste del transformador en X_fit."""
    global_keys = ("X_train", "y_train", "X_test", "y_test", "escaladores",
                   "modelos_originales", "modelos_nuevos", "modelos_hibridos",
                   "ESTADO_COMPARACION", "tabla_resultados", "tabla_validacion_ultima",
                   "crear_pipeline_santander",
                   "_PREPROCESAMIENTO_ULTIMO_ORIGEN", "_HOLDOUT_ACTIVE_TOKEN")
    old_globals = {key: globals().get(key) for key in global_keys}
    existed = {key: key in globals() for key in global_keys}
    old_experiment_state = dict(ESTADO_COMPARACION)
    old_results = list(resultados)
    old_val_models = dict(modelos_validacion)
    old_hold_models = dict(modelos_holdout)
    old_predictions = dict(predicciones_holdout)
    try:
        # Dataset artificial mínimo, usado solo como fixture para detectar fugas y errores de API.
        Xtr = pd.DataFrame({"x": [0., 2., 20., 22., 24., 10000.]}, index=range(6))
        ytr = pd.Series([0, 0, 1, 1, 0, 1], index=Xtr.index)
        Xte = pd.DataFrame({"x": [20000., 20002.]}, index=[6, 7])
        yte = pd.Series([0, 1], index=Xte.index)
        llamadas_poma = []
        factory_poma_real = globals().get("crear_pipeline_santander")
        if not callable(factory_poma_real):
            raise AssertionError("La prueba requiere la fábrica real de pipeline de Poma definida en 4.1.3.")
        def factory_poma(clasificador, escalador=None):
            llamadas_poma.append((type(clasificador).__name__, escalador is not None))
            return factory_poma_real(clasificador, escalador=escalador)
        globals().update({"X_train": Xtr, "y_train": ytr, "X_test": Xte, "y_test": yte,
            "escaladores": {"standard": StandardScaler()},
            "modelos_originales": {"__prueba_tecnica__": LogisticRegression(solver="liblinear", random_state=42)},
            "modelos_nuevos": {}, "modelos_hibridos": {},
            "crear_pipeline_santander": factory_poma,
            "_PREPROCESAMIENTO_ULTIMO_ORIGEN": None, "_HOLDOUT_ACTIVE_TOKEN": None,
            "ESTADO_COMPARACION": {"revision": 0, "validacion": pd.DataFrame(), "modelos": {},
                "escaladores": {}, "holdout_status": "sin_experimento",
                "evaluada_revision": None, "seleccion_holdout_evaluada": None}})
        estimator = LogisticRegression(solver="liblinear", random_state=42)
        Xfit, yfit = Xtr.iloc[:4], ytr.iloc[:4]
        Xval, yval = Xtr.iloc[4:], ytr.iloc[4:]

        # Prueba negativa: omitir particiones explícitas debe fallar en la llamada.
        try:
            evaluar_modelo("__prueba__", "original", estimator)
        except TypeError:
            pass
        else:
            raise AssertionError("La interfaz aceptó una llamada sin particiones explícitas.")

        # Prueba funcional: el escalador usa las cuatro filas de Xfit, nunca las dos de Xval.
        metricas = evaluar_modelo("__prueba__", "original", estimator, StandardScaler(),
            X_fit=Xfit, y_fit=yfit, X_eval=Xval, y_eval=yval,
            etapa="validacion", guardar=False, guardar_ajustado=True, fold=1)
        fitted = list(modelos_validacion.values())[-1]
        score = _puntaje_continuo(fitted, Xval)
        auc_score = roc_auc_score(yval, score)
        auc_etiquetas = roc_auc_score(yval, fitted.predict(Xval))
        if not np.isclose(metricas["roc_auc"], auc_score):
            raise AssertionError("ROC-AUC no se calculó a partir de scores continuos.")
        if np.isclose(auc_score, auc_etiquetas):
            raise AssertionError("El fixture técnico no distingue score continuo de etiqueta predicha.")
        if not llamadas_poma or not _PREPROCESAMIENTO_ULTIMO_ORIGEN.startswith("Poma:"):
            raise AssertionError("La fábrica de pipeline completo de Poma fue ignorada.")
        if [name for name, _ in fitted.steps].count("escalador") != 1:
            raise AssertionError("Se duplicó o perdió el escalador compartido.")
        scaler_fit = fitted.named_steps["escalador"]
        if not np.allclose(scaler_fit.mean_, [Xfit["x"].mean()]):
            raise AssertionError("El transformador no se ajustó exclusivamente con X_fit.")
        if np.allclose(scaler_fit.mean_, [pd.concat([Xfit["x"], Xval["x"]]).mean()]):
            raise AssertionError("El transformador absorbió los valores del fold de validación.")

        # Pruebas negativas: alteraciones en filas/etiquetas se rechazan aunque conserven índices.
        X_bad = Xfit.copy()
        X_bad.iloc[0, 0] += 999
        y_bad = yval.copy()
        y_bad.iloc[0] = 1 - y_bad.iloc[0]
        for bad_xfit, bad_yval in [(X_bad, yval), (Xfit, y_bad)]:
            try:
                evaluar_modelo("__procedencia__", "original", estimator, StandardScaler(),
                    X_fit=bad_xfit, y_fit=yfit, X_eval=Xval, y_eval=bad_yval,
                    etapa="validacion", guardar=False)
            except ValueError:
                pass
            else:
                raise AssertionError("La interfaz aceptó datos/etiquetas que no corresponden al origen.")
        try:
            evaluar_modelo("__holdout_directo__", "original", estimator, StandardScaler(),
                X_fit=Xtr, y_fit=ytr, X_eval=Xte, y_eval=yte, etapa="holdout", guardar=False)
        except RuntimeError:
            pass
        else:
            raise AssertionError("La interfaz permitió holdout fuera del flujo controlado.")

        # Prueba negativa: X_test/y_test no se pueden usar con etapa='validacion'.
        try:
            evaluar_modelo("__negativa__", "original", estimator, StandardScaler(),
                X_fit=Xfit, y_fit=yfit, X_eval=Xte, y_eval=yte,
                etapa="validacion", guardar=False)
        except ValueError as exc:
            if "x_train" not in str(exc).lower() and "holdout" not in str(exc).lower():
                raise AssertionError(f"El holdout se rechazó por un motivo inesperado: {exc}")
        else:
            raise AssertionError("La interfaz aceptó el holdout como conjunto de validación.")

        # La CV no usa valores/etiquetas del holdout; solo se verifica disjunción de índices.
        globals()["X_test"] = pd.DataFrame({"centinela": [-1., -2.]}, index=[6, 7])
        globals()["y_test"] = pd.Series([1, 1], index=[6, 7])
        cv_table = ejecutar_comparacion(modo="completo", n_splits=2, reiniciar_experimento=True)
        published_cv = globals().get("tabla_validacion_ultima")
        published_rows = globals().get("tabla_resultados")
        if cv_table.empty or not (cv_table["estado"] == "OK").all():
            raise AssertionError("La CV técnica no publicó configuraciones válidas.")
        if published_cv is None or published_rows is None:
            raise AssertionError("Las tablas compartidas no se publicaron automáticamente.")
        if not set(published_rows["etapa"]) <= {"validacion"}:
            raise AssertionError("El comparador ejecutó el holdout durante la exploración CV.")
        if ESTADO_COMPARACION["holdout_status"] != "cv_disponible":
            raise AssertionError("La exploración CV alteró el estado del holdout.")

        # Repetir CV sin reset explícito debe fallar y dejar intacta la revisión.
        revision_cv = ESTADO_COMPARACION["revision"]
        try:
            ejecutar_comparacion(modo="completo", n_splits=2)
        except RuntimeError:
            pass
        else:
            raise AssertionError("Repetir el comparador habilitó otro experimento sin reset.")
        if ESTADO_COMPARACION["revision"] != revision_cv:
            raise AssertionError("La repetición bloqueada modificó la revisión CV.")

        # La evaluación final explícita ajusta el escalador en el entrenamiento completo.
        globals()["X_test"], globals()["y_test"] = Xte, yte
        final = evaluar_holdout_final()
        if final.empty or set(final["etapa"]) != {"holdout"}:
            raise AssertionError("La interfaz final no publicó filas de holdout.")
        if ESTADO_COMPARACION["evaluada_revision"] != revision_cv or not ESTADO_COMPARACION["seleccion_holdout_evaluada"]["configuraciones"]:
            raise AssertionError("No quedó registrada la selección CV usada para el holdout.")
        fitted_final = list(modelos_holdout.values())[-1]
        scaler_final = fitted_final.named_steps["escalador"]
        if not np.allclose(scaler_final.mean_, [Xtr["x"].mean()]):
            raise AssertionError("El transformador final se ajustó fuera de X_train.")
        try:
            evaluar_modelo("__holdout_repetido_directo__", "original", estimator, StandardScaler(),
                X_fit=Xtr, y_fit=ytr, X_eval=Xte, y_eval=yte, etapa="holdout", guardar=False)
        except RuntimeError:
            pass
        else:
            raise AssertionError("Una segunda llamada directa a evaluar_modelo aceptó el holdout.")
        try:
            evaluar_holdout_final()
        except RuntimeError:
            pass
        else:
            raise AssertionError("La API permitió evaluar el mismo holdout dos veces.")
        try:
            ejecutar_comparacion(modo="completo", n_splits=2)
        except RuntimeError:
            pass
        else:
            raise AssertionError("El holdout evaluado no bloqueó una repetición del comparador.")
        reset_table = ejecutar_comparacion(modo="completo", n_splits=2, reiniciar_experimento=True)
        if reset_table.empty or ESTADO_COMPARACION["revision"] != revision_cv + 1:
            raise AssertionError("El reinicio explícito no creó una nueva revisión CV.")
        if ESTADO_COMPARACION["holdout_status"] != "cv_disponible":
            raise AssertionError("El reinicio explícito no dejó un nuevo holdout pendiente.")
        if _validar_esquema_resultados(pd.DataFrame({"accuracy": [0.5]}))[0] != "FAIL":
            raise AssertionError("El esquema incompleto no se registró como FAIL.")
        if _validar_esquema_resultados(pd.DataFrame())[0] != "FAIL":
            raise AssertionError("Una tabla vacía sin esquema no se registró como FAIL.")
        if _validar_esquema_cv(pd.DataFrame({"estado": ["OK"]}))[0] != "FAIL":
            raise AssertionError("El esquema CV incompleto no se registró como FAIL.")
        return "PASS: pipeline completo de Poma, procedencia X/y, escalado único, gate holdout, reset CV explícito, esquema y ROC. Fixture técnico; sin resultados científicos."
    finally:
        resultados[:] = old_results
        modelos_validacion.clear(); modelos_validacion.update(old_val_models)
        modelos_holdout.clear(); modelos_holdout.update(old_hold_models)
        predicciones_holdout.clear(); predicciones_holdout.update(old_predictions)
        ESTADO_COMPARACION.clear(); ESTADO_COMPARACION.update(old_experiment_state)
        for key in global_keys:
            if existed[key]:
                globals()[key] = old_globals[key]
            else:
                globals().pop(key, None)

def auditar_integracion():
    filas = []
    def reportar(categoria, nombre, estado, detalle):
        filas.append({"categoria": categoria, "verificacion": nombre,
                      "estado": estado, "detalle": str(detalle)})
    def comprobar_dic(nombre, esperadas, marco=False):
        value = globals().get(nombre)
        if not isinstance(value, dict):
            reportar("estructural", nombre, "FAIL" if marco else "PENDIENTE", "Diccionario aún no definido.")
            return
        missing = [key for key in esperadas if key not in value]
        extra = [key for key in value if key not in esperadas]
        state = "FAIL" if extra or (missing and marco) else ("PENDIENTE" if missing else "PASS")
        reportar("estructural", nombre, state, f"{len(value)} entradas; faltan={missing}; extras={extra}")
    comprobar_dic("escaladores", ["standard", "minmax", "robust", "quantile"])
    comprobar_dic("modelos_originales", ["random_forest", "decision_tree", "xgboost", "catboost", "gaussian_nb", "logistic_regression"], True)
    comprobar_dic("modelos_nuevos", ["svm_lineal", "adaboost", "knn", "lightgbm", "extra_trees"])
    comprobar_dic("modelos_hibridos", ["H1_voting_hard", "H2_voting_soft", "H3_voting_soft_boosting", "H4_stacking_svm", "H5_stacking_arboles"])

    Xtr, Xte = globals().get("X_train"), globals().get("X_test")
    ytr, yte = globals().get("y_train"), globals().get("y_test")
    if not all(v is not None for v in (Xtr, Xte, ytr, yte)):
        reportar("estructural", "split_datos", "PENDIENTE", "Falta ejecutar la carga/división de Poma.")
    else:
        reportar("estructural", "ID_predictor", "FAIL" if "ID" in Xtr.columns or "ID" in Xte.columns else "PASS",
                 "ID no debe aparecer entre predictores.")
        reportar("estructural", "columnas", "PASS" if list(Xtr.columns) == list(Xte.columns) else "FAIL",
                 "Mismas columnas y orden para train y holdout.")
        overlap = bool(Xtr.index.intersection(Xte.index).size)
        reportar("estructural", "split_sin_solapamiento", "FAIL" if overlap else "PASS",
                 "Índices train/holdout disjuntos." if not overlap else "Índices compartidos.")
        if not isinstance(Xtr, pd.DataFrame) or not isinstance(Xte, pd.DataFrame) or not isinstance(ytr, pd.Series) or not isinstance(yte, pd.Series):
            reportar("estructural", "split_esquema", "FAIL", "Se requieren DataFrame/Series para X_train/X_test/y_train/y_test.")
        else:
            binary = set(ytr.dropna().unique()) == {0, 1} and set(yte.dropna().unique()) == {0, 1}
            reportar("estructural", "etiquetas_binarias", "PASS" if binary else "FAIL", "Ambas clases deben estar presentes.")

    if isinstance(Xtr, pd.DataFrame) and callable(globals().get("_crear_pipeline_evaluacion")):
        estimators = []
        for dict_name in ("modelos_originales", "modelos_nuevos", "modelos_hibridos"):
            values = globals().get(dict_name)
            if isinstance(values, dict):
                estimators.extend(model for model in values.values() if model is not None)
        try:
            for estimator in estimators:
                clone(_crear_pipeline_evaluacion(estimator, Xtr))
            state = "PASS" if estimators else "PENDIENTE"
            reportar("estructural", "pipelines_clonables", state,
                     f"{len(estimators)} modelos clonables." if estimators else "Aún no hay modelos disponibles.")
        except Exception as exc:
            reportar("estructural", "pipelines_clonables", "FAIL", f"{type(exc).__name__}: {exc}")
    else:
        reportar("estructural", "pipelines_clonables", "PENDIENTE", "Requiere X_train y fábrica de Pipeline.")

    try:
        detail = _prueba_funcional_pipeline_y_holdout()
        reportar("funcional", "aislamiento_pipeline_holdout", "PASS", detail)
    except Exception as exc:
        reportar("funcional", "aislamiento_pipeline_holdout", "FAIL", f"{type(exc).__name__}: {exc}")

    routes = globals().get("RUTAS_CSV")
    if not isinstance(routes, dict):
        reportar("estructural", "archivos_csv", "PENDIENTE", "Falta ejecutar la configuración de entorno.")
    else:
        missing = [name for name, route in routes.items() if not Path(route).is_file()]
        reportar("estructural", "archivos_csv", "PENDIENTE" if missing else "PASS",
                 f"Faltan: {missing}" if missing else "CSV localizados.")

    pipeline_factory = globals().get("crear_pipeline_santander")
    if pipeline_factory is None:
        reportar("estructural", "pipeline_Poma", "PENDIENTE",
                 "La fábrica compartida no está disponible; Marco usa fallback local informado.")
    elif not callable(pipeline_factory):
        reportar("estructural", "pipeline_Poma", "FAIL", "crear_pipeline_santander no es callable.")
    else:
        try:
            import inspect
            inspect.signature(pipeline_factory).bind(LogisticRegression(), escalador=None)
            reportar("estructural", "pipeline_Poma", "PASS",
                     "Firma crear_pipeline_santander(clasificador, escalador=None) disponible.")
        except (TypeError, ValueError) as exc:
            reportar("estructural", "pipeline_Poma", "FAIL", f"Firma incompatible: {exc}")
    reportar("estructural", "remuestreo", "PASS",
             "No se solicita remuestreo; coincide con la implementación aprobada de Poma.")

    result_table = globals().get("tabla_resultados")
    result_state, result_detail = _validar_esquema_resultados(result_table)
    reportar("estructural", "integridad_metricas", result_state, result_detail)
    cv_table = globals().get("tabla_validacion_ultima")
    cv_state, cv_detail = _validar_esquema_cv(cv_table)
    reportar("estructural", "integridad_tabla_CV", cv_state, cv_detail)
    colab = bool(globals().get("EN_COLAB"))
    reportar("estructural", "compatibilidad_colab", "PASS" if colab else "PENDIENTE",
             "Entorno Colab detectado." if colab else "Auditoría ejecutada fuera de Colab.")
    report = pd.DataFrame(filas)
    display(report)
    return report

auditoria_integracion = auditar_integracion()

## 4.7. Lista de comprobación del equipo antes de entregar

- [ ] **Marco:** los tres CSV están accesibles en Colab y el protocolo está definido.
- [ ] **Poma:** split antes de ajustes/remuestreo; existen 4 escaladores definidos sin fugas.
- [ ] **Marco:** seis modelos base comparados de forma válida.
- [ ] **Aldana:** cinco modelos nuevos y sus métricas están registrados.
- [ ] **Brickzon:** H1, H2 y H3 se construyen y se evalúan correctamente.
- [ ] **Vera:** H4 y H5 se construyen; cinco híbridos auditados.
- [ ] **Marco:** resultados, gráficos, conclusiones y ejecución limpia de Parte B.
- [ ] **Equipo:** se documentaron limitaciones, versiones y combinaciones omitidas si existieron.

> **Estado de esta plantilla:** planificación y celdas asignadas, **sin implementaciones nuevas ni resultados calculados**.